In [ ]:
%pip install ipympl  
%matplotlib widget

# Data loading

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import glob
from scipy.signal import medfilt
from pathlib import Path
from tqdm import tqdm

try:
    import pint as _pint
except ModuleNotFoundError:
    _pint = None
else:
    _original_unit_registry = _pint.UnitRegistry

    def _unit_registry_without_disk_cache(*args, **kwargs):
        if kwargs.get("cache_folder") == ":auto:":
            kwargs["cache_folder"] = None
        return _original_unit_registry(*args, **kwargs)

    _pint.UnitRegistry = _unit_registry_without_disk_cache

try:
    import dascore as dc
except ModuleNotFoundError as exc:
    if exc.name != "dascore":
        raise
    dascore_repo = Path(r"D:\OneDrive - Colorado School of Mines\GitHub\dascore")
    if not dascore_repo.exists():
        raise ModuleNotFoundError(
            "DASCore is not installed and the cloned repository was not found at "
            f"{dascore_repo}"
        ) from exc
    sys.path.insert(0, str(dascore_repo))
    import dascore as dc

tqdm.pandas()

# === USER CONFIGURATION ===
# start_time = pd.to_datetime('2025-02-21 00:00')
# end_time   = pd.to_datetime('2025-02-28 00:00')
# data_path = Path("D:/OneDrive - Colorado School of Mines/Research/Fervo/Aleksei/Data/Gold 4-PB/DAS/LF-DAS NPY/NPY_G4-PB_MM_UTC_202502")

start_time = pd.to_datetime('2025-02-23 00:00')  # Start datetime for filtering data
end_time   = pd.to_datetime('2025-03-05 00:00')    # End datetime for filtering data
#data_path_02 = Path(r"C:\Users\13811\OneDrive - Colorado School of Mines\Research\Fervo\Aleksei\Data\Gold 4-PB\DAS\LF-DAS NPY\NPY_G4-PB_MM_UTC_202502")
#data_path_03 = Path(r"C:\Users\13811\OneDrive - Colorado School of Mines\Research\Fervo\Aleksei\Data\Gold 4-PB\DAS\LF-DAS NPY\NPY_G4-PB_MM_UTC_202503")
data_path_02 = Path(r"D:\OneDrive - Colorado School of Mines\Research\Fervo\Aleksei\Data\Gold 4-PB\DAS\LF-DAS NPY\NPY_G4-PB_MM_UTC_202502")
data_path_03 = Path(r"D:\OneDrive - Colorado School of Mines\Research\Fervo\Aleksei\Data\Gold 4-PB\DAS\LF-DAS NPY\NPY_G4-PB_MM_UTC_202503")

TIME_SAMPLE_OFFSETS_SECONDS = np.arange(5, 60, 10)
EXPECTED_TIME_STEP = pd.Timedelta(seconds=10)
LOCAL_TIME_OFFSET = pd.Timedelta(hours=7)
fs = 1000
GL = 10
STRAIN_RATE_SCALE = 116 * fs / GL / 8192


def load_npy_as_dascore_patch(file_paths, time_zone_label="UTC-7"):
    records = []
    for file_path in tqdm(file_paths, desc="Loading NPY files"):
        file_path = Path(file_path)
        file_start = pd.to_datetime(
            file_path.stem.split("UTC_")[-1],
            format="%Y%m%d_%H%M%S",
        )
        data = np.load(file_path) * STRAIN_RATE_SCALE
        if data.shape[0] != len(TIME_SAMPLE_OFFSETS_SECONDS):
            raise ValueError(
                f"Expected {len(TIME_SAMPLE_OFFSETS_SECONDS)} time samples in {file_path.name}, "
                f"but found shape {data.shape}."
            )
        times = np.array(
            [file_start + pd.Timedelta(seconds=int(offset)) for offset in TIME_SAMPLE_OFFSETS_SECONDS],
            dtype="datetime64[ns]",
        )
        records.append((file_path, times, data))

    if not records:
        raise FileNotFoundError("No NPY files were found in the requested time range.")

    paths, time_blocks, data_blocks = zip(*records)
    time_utc = pd.DatetimeIndex(np.concatenate(time_blocks))
    time_local = time_utc - LOCAL_TIME_OFFSET
    data_time_channel = np.concatenate(data_blocks, axis=0)
    distance = np.arange(data_time_channel.shape[1])

    patch = dc.Patch(
        data=data_time_channel.T,
        coords={"distance": distance, "time": time_local.to_numpy(dtype="datetime64[ns]")},
        dims=("distance", "time"),
        attrs={
            "data_type": "low-frequency DAS strain rate",
            "data_units": "nanostrain/s",
            "time_zone": time_zone_label,
            "source_file_count": len(paths),
        },
    )
    return patch, pd.DataFrame({"path": [str(path) for path in paths], "dt": [times[0] for times in time_blocks]})


def fill_dascore_time_gaps_with_nan(patch, expected_step=EXPECTED_TIME_STEP):
    time_index = pd.DatetimeIndex(patch.get_coord("time").values)
    time_diffs = pd.Series(time_index).diff().dt.total_seconds()
    expected_seconds = expected_step.total_seconds()

    gap_info = pd.DataFrame(
        {
            "gap_start": pd.Series(time_index).shift(),
            "gap_end": pd.Series(time_index),
            "gap_seconds": time_diffs,
        }
    ).dropna()
    gap_info = gap_info[gap_info["gap_seconds"] > expected_seconds].copy()
    gap_info["inserted_nan_samples"] = (
        np.round(gap_info["gap_seconds"] / expected_seconds).astype(int) - 1
    ).clip(lower=0)

    filled_times = []
    filled_columns = []
    data = np.asarray(patch.data)

    for idx, current_time in enumerate(time_index):
        if idx:
            previous_time = time_index[idx - 1]
            gap_seconds = (current_time - previous_time).total_seconds()
            inserted_count = max(0, int(round(gap_seconds / expected_seconds)) - 1)
            for step_index in range(1, inserted_count + 1):
                filled_times.append(previous_time + step_index * expected_step)
                filled_columns.append(np.full(data.shape[0], np.nan, dtype=data.dtype))
        filled_times.append(current_time)
        filled_columns.append(data[:, idx])

    filled_data = np.column_stack(filled_columns)
    filled_time_index = pd.DatetimeIndex(filled_times)
    filled_patch = dc.Patch(
        data=filled_data,
        coords={
            "distance": patch.get_coord("distance").values,
            "time": filled_time_index.to_numpy(dtype="datetime64[ns]"),
        },
        dims=patch.dims,
        attrs=dict(patch.attrs) | {"time_gaps_filled_with": "NaN"},
    )
    return filled_patch, gap_info.reset_index(drop=True)


# List files and keep only the requested datetime range.
file_list_02 = sorted(glob.glob(str(data_path_02 / '*.npy'), recursive=True))
file_list_03 = sorted(glob.glob(str(data_path_03 / '*.npy'), recursive=True))
file_list = file_list_02 + file_list_03

df_files = pd.DataFrame(file_list, columns=['path'])
df_files['dt'] = df_files.path.apply(lambda x: pd.to_datetime(Path(x).stem.split("UTC_")[-1], format='%Y%m%d_%H%M%S'))
df_files = df_files.query("@start_time <= dt <= @end_time").sort_values('dt').reset_index(drop=True)

das_patch_raw, df_loaded_files = load_npy_as_dascore_patch(df_files['path'])
das_patch, das_time_gap_report = fill_dascore_time_gaps_with_nan(das_patch_raw)

LP = np.asarray(das_patch.data)
Time_raw = pd.DatetimeIndex(das_patch.get_coord("time").values)

print(f"Using DASCore {dc.__version__}")
print(f"Loaded {len(df_loaded_files):,} NPY files into a DASCore Patch.")
print(f"Raw DAS samples: {das_patch_raw.data.shape[1]:,}; after gap filling: {das_patch.data.shape[1]:,}")
print(f"Detected {len(das_time_gap_report)} timeline gaps > {EXPECTED_TIME_STEP}.")
if not das_time_gap_report.empty:
    display(das_time_gap_report)


# Keep full-column NaNs inserted for missing DAS time samples visible in waterfall plots.
timeline_gap_mask = np.isnan(LP).all(axis=0)


def gap_aware_cmap(cmap_name="bwr", bad_color="0.82"):
    cmap = plt.get_cmap(cmap_name).copy() if isinstance(cmap_name, str) else cmap_name.copy()
    cmap.set_bad(color=bad_color)
    return cmap


def make_dascore_waterfall_patch(
    data,
    time,
    depth_start,
    depth_end,
    data_type="DAS strain rate",
    data_units="nanostrain/s",
):
    data = np.asarray(data)
    time = pd.DatetimeIndex(time).to_numpy(dtype="datetime64[ns]")
    measured_depth = np.linspace(depth_start, depth_end, data.shape[0])
    return dc.Patch(
        data=data,
        coords={"measured_depth": measured_depth, "time": time},
        dims=("measured_depth", "time"),
        attrs={
            "data_type": data_type,
            "data_units": data_units,
            "time_zone": "UTC-7",
            "time_gaps_filled_with": "NaN",
        },
    )


def plot_dascore_waterfall(
    ax,
    data,
    time,
    depth_start,
    depth_end,
    cmap="bwr",
    vmin=None,
    vmax=None,
    cbar=False,
    data_type="DAS strain rate",
    data_units="nanostrain/s",
):
    patch = make_dascore_waterfall_patch(
        data=data,
        time=time,
        depth_start=depth_start,
        depth_end=depth_end,
        data_type=data_type,
        data_units=data_units,
    )
    scale = None if (vmin is None and vmax is None) else (vmin, vmax)
    ax = patch.viz.waterfall(
        ax=ax,
        cmap=gap_aware_cmap(cmap),
        scale=scale,
        scale_type="absolute",
        interpolation=None,
        cbar=cbar,
        show=False,
    )
    ax.set_ylabel("Gold 4-PB Measured Depth [ft]")
    ax.set_xlabel("Time [UTC-7]")
    return ax


print(f"DASCore waterfall plots will preserve {int(timeline_gap_mask.sum())} DAS time-gap columns as blank/gray bands.")


bowf, eowf, bowmd, eowmd = 52, 2255, 15, 14807
a = (eowmd - bowmd) / (eowf - bowf)
lead = -a*(bowf - bowmd/a)
eor = 2304*a + lead






In [ ]:
'''
In your code, MD is calculated by calibrating DAS channel index to well survey data:

bowf, eowf, bowmd, eowmd = 52, 2255, 15, 14807

a = (eowmd - bowmd) / (eowf - bowf)   # slope (ft per channel)
lead = -a * (bowf - bowmd/a)          # intercept
channels = np.arange(LP.shape[0])     # DAS channels
md_axis = a * channels + lead         # measured depth (ft)


bowf = beginning of well fiber channel (index along fiber).

eowf = end of well fiber channel.

bowmd = measured depth corresponding to bowf (from well survey).

eowmd = measured depth corresponding to eowf.

So the mapping is a simple linear transformation:

MD=a鈰卌hannel+lead
MD=a鈰卌hannel+lead

where a is the scaling factor (ft/channel) and lead is the offset.
'''

# waterfall map plotting

In [ ]:
st_md = 1800
ed_md = 5000
st_idx = int((st_md - lead) / a)
ed_idx = int((ed_md - lead) / a)
LP_med = medfilt(LP, kernel_size=[3,7])
LP_demean = LP_med - np.nanmedian(LP_med[st_idx:ed_idx, :], axis=0)

# Fill isolated NaNs inside valid data, but keep full-column timeline gaps blank in waterfall plots.
timeline_gap_mask = np.isnan(LP).all(axis=0)
for i in range(LP_demean.shape[0]):
    y = LP_demean[i, :]
    nans_to_fill = np.isnan(y) & ~timeline_gap_mask
    valid = ~np.isnan(y) & ~timeline_gap_mask
    if np.any(nans_to_fill) and np.count_nonzero(valid) >= 2:
        y[nans_to_fill] = np.interp(
            np.flatnonzero(nans_to_fill),
            np.flatnonzero(valid),
            y[valid],
        )
    y[timeline_gap_mask] = np.nan
    LP_demean[i, :] = y


# Remove a narrow vertical artifact attributed to interrogator vibration.
# This does not remove DASCore gap filling; true timeline-gap columns remain NaN.
INTERROGATOR_ARTIFACT_WINDOWS = [
    (pd.Timestamp("2025-02-25 19:53:50"), pd.Timestamp("2025-02-25 19:54:30")),
]


def remove_vertical_time_artifacts(data, time_index, artifact_windows, keep_nan_mask=None):
    cleaned = np.asarray(data, dtype=float).copy()
    time_index = pd.DatetimeIndex(time_index)
    artifact_mask = np.zeros(len(time_index), dtype=bool)

    for start, end in artifact_windows:
        artifact_mask |= (time_index >= start) & (time_index <= end)

    if keep_nan_mask is not None:
        artifact_mask &= ~keep_nan_mask

    x = np.arange(cleaned.shape[1])
    for row_idx in range(cleaned.shape[0]):
        y = cleaned[row_idx, :]
        valid = np.isfinite(y) & ~artifact_mask
        if np.any(artifact_mask) and np.count_nonzero(valid) >= 2:
            y[artifact_mask] = np.interp(x[artifact_mask], x[valid], y[valid])
        cleaned[row_idx, :] = y

    if keep_nan_mask is not None:
        cleaned[:, keep_nan_mask] = np.nan

    return cleaned, artifact_mask


LP_demean, interrogator_artifact_mask = remove_vertical_time_artifacts(
    LP_demean,
    Time_raw,
    INTERROGATOR_ARTIFACT_WINDOWS,
    keep_nan_mask=timeline_gap_mask,
)
print(
    f"Removed/interpolated {int(interrogator_artifact_mask.sum())} interrogator-vibration "
    "time columns; DASCore gap-filled columns remain NaN."
)

################################################
from matplotlib.pyplot import figure, imshow
figure(figsize=(15, 6))
plt.xlabel('Time (UTC-7)')
extent = (Time_raw[0].to_pydatetime(), Time_raw[-1].to_pydatetime(), eor, lead)
imshow(LP_demean, cmap='bwr', aspect='auto',
                 vmin=-2, vmax=2, extent=extent)
plt.ylabel('Measured Depth (ft)')
plt.title('DAS Heatmap (Gold 4-PB)')
plt.colorbar(label='Strain Rate (microstrain/s)')
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%m-%d\n%H:%M'))
plt.ylim(12250,9500)
plt.show()



# waterfall map plotting using DASCore

In [ ]:
# DASCore-native waterfall map plotting
# This cell uses DASCore's Patch.viz.waterfall directly.

Time_raw_dt = pd.DatetimeIndex(Time_raw)
depth_axis = np.linspace(lead, eor, LP_demean.shape[0])

das_waterfall_patch = dc.Patch(
    data=LP_demean,
    coords={
        "measured_depth": depth_axis,
        "time": Time_raw_dt.to_numpy(dtype="datetime64[ns]"),
    },
    dims=("measured_depth", "time"),
    attrs={
        "data_type": "DAS strain rate",
        "data_units": "nanostrain/s",
        "time_zone": "UTC-7",
        "time_gaps_filled_with": "NaN",
    },
)

das_cmap = plt.get_cmap("bwr").copy()
das_cmap.set_bad("0.82")  # show inserted timeline gaps as light-gray bands

fig, ax = plt.subplots(figsize=(15, 6))
ax = das_waterfall_patch.viz.waterfall(
    ax=ax,
    cmap=das_cmap,
    scale=(-2, 2),
    scale_type="absolute",
    interpolation=None,
    cbar=True,
    show=False,
)

ax.set_title("DASCore Waterfall Map (Gold 4-PB)")
ax.set_xlabel("Time (UTC-7)")
ax.set_ylabel("Measured Depth (ft)")
ax.set_ylim(12250, 9500)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d\n%H:%M"))

# DASCore displays missing samples from NaN data only; no extra gray overlay is added.

plt.show()





# load pumping data

In [ ]:
# Load pumping / WHP data
def load_whp(path, label):
    df = pd.read_csv(path)
    df["time"] = pd.to_datetime(df["MSTTIMESTAMP"])
    df = df.query("@start_time <= time <= @end_time").copy()
    df["pressure"] = pd.to_numeric(df["PRESSURE_PSI"], errors="coerce")
    df["stage"] = pd.to_numeric(df.get("STAGENUMBER", np.nan), errors="coerce")
    df = df.dropna(subset=["time", "pressure"])
    df["well"] = label
    return df[["time", "pressure", "stage", "well"]]


whp_files = {
    "1-IA": r"D:/OneDrive - Colorado School of Mines/Research/Fervo/Aleksei/Data/Bearskin Stimulation Data/Bearskin 1-IA Pressure.csv",
    "3-PA": r"D:/OneDrive - Colorado School of Mines/Research/Fervo/Aleksei/Data/Bearskin Stimulation Data/Bearskin 3-PA Pressure.csv",
    "4-PB": r"D:/OneDrive - Colorado School of Mines/Research/Fervo/Aleksei/Data/Bearskin Stimulation Data/Bearskin 4-PB Pressure.csv",
}

whp_df = pd.concat(
    [load_whp(path, f"Bearskin {well} WHP") for well, path in whp_files.items()],
    ignore_index=True,
)

pumping_windows_path = r"D:/OneDrive - Colorado School of Mines/Research/Fervo/Pengchao/input/pumping_windows.xlsx"
pumping_df = pd.read_excel(pumping_windows_path)
pumping_df = pumping_df[["Start Time (AM/PM)", "End Time (AM/PM)"]].copy()
pumping_df.columns = ["start", "end"]
pumping_df["start"] = pd.to_datetime(pumping_df["start"])
pumping_df["end"] = pd.to_datetime(pumping_df["end"])
pumping_df["center"] = pumping_df["start"] + (pumping_df["end"] - pumping_df["start"]) / 2

injection_path = r"D:/OneDrive - Colorado School of Mines/Research/Fervo/Pengchao/input/injection_projected_along_strike.csv"
injection_df = pd.read_csv(injection_path)
injection_df["start"] = pd.to_datetime(injection_df["Start Time (AM/PM)"])
injection_df["end"] = pd.to_datetime(injection_df["End Time (AM/PM)"])
injection_df["center"] = injection_df["start"] + (injection_df["end"] - injection_df["start"]) / 2

print(f"Loaded {len(whp_df):,} WHP samples, {len(pumping_df)} pumping windows, and {len(injection_df)} injection intervals.")



# plot pumping data on top of waterfall map

In [ ]:
# Plot pumping data on top of DASCore waterfall map
# Uses the same measured-depth range as the "# DASCore-native waterfall map plotting" cell.

plot_start = pd.to_datetime("2025-02-23 00:00")
plot_end = pd.to_datetime("2025-03-04 17:00")
plot_md_top = 9500
plot_md_bottom = 12250

# Time markers used to compare pumping stages with the DAS response.
time_marks = pd.to_datetime([
    "2025-02-24 11:00",
    "2025-02-28 00:00",
    "2025-03-03 22:00",
])
mark_labels = ["T1", "T2", "T3"]

Time_raw_dt = pd.DatetimeIndex(Time_raw)
depth_axis = np.linspace(lead, eor, LP_demean.shape[0])

time_mask = (Time_raw_dt >= plot_start) & (Time_raw_dt <= plot_end)
md_mask = (depth_axis >= plot_md_top) & (depth_axis <= plot_md_bottom)

Time_plot = Time_raw_dt[time_mask]
Depth_plot = depth_axis[md_mask]
LP_plot = LP_demean[md_mask, :][:, time_mask]

das_pump_patch = dc.Patch(
    data=LP_plot,
    coords={
        "measured_depth": Depth_plot,
        "time": Time_plot.to_numpy(dtype="datetime64[ns]"),
    },
    dims=("measured_depth", "time"),
    attrs={
        "data_type": "DAS strain rate",
        "data_units": "nanostrain/s",
        "time_zone": "UTC-7",
        "time_gaps_filled_with": "NaN",
    },
)

das_cmap = plt.get_cmap("bwr").copy()
das_cmap.set_bad("0.82")

fig, (ax1, ax2) = plt.subplots(
    2,
    1,
    figsize=(14, 6),
    sharex=True,
    gridspec_kw=dict(height_ratios=[1, 2.5]),
    constrained_layout=True,
)

color_map = {
    "Bearskin 1-IA WHP": "orange",
    "Bearskin 3-PA WHP": "red",
    "Bearskin 4-PB WHP": "purple",
}

# Top panel: pumping / WHP curves only.
for well, grp in whp_df.groupby("well"):
    ax1.plot(
        grp["time"],
        grp["pressure"],
        label=well,
        linewidth=1.5,
        color=color_map.get(well, "black"),
    )

# Label pumping stages on the Bearskin 4-PB WHP curve.
stage_source = whp_df[(whp_df["well"] == "Bearskin 4-PB WHP") & whp_df["stage"].notna()].copy()
if not stage_source.empty:
    stage_source = stage_source.sort_values("time")
    stage_source["segment_id"] = (
        stage_source["stage"].ne(stage_source["stage"].shift())
        | stage_source["time"].diff().gt(pd.Timedelta(minutes=1.1))
    ).cumsum()
    stage_segments = (
        stage_source.groupby("segment_id")
        .agg(
            stage=("stage", "first"),
            start=("time", "min"),
            end=("time", "max"),
            ymax=("pressure", "max"),
        )
        .reset_index(drop=True)
    )
    pressure_pad = max(50, 0.03 * (stage_source["pressure"].max() - stage_source["pressure"].min()))
    for _, seg in stage_segments.iterrows():
        center_time = seg["start"] + (seg["end"] - seg["start"]) / 2
        ax1.text(
            center_time,
            seg["ymax"] + pressure_pad,
            f"{int(seg['stage'])}",
            color=color_map["Bearskin 4-PB WHP"],
            fontsize=9,
            fontweight="bold",
            ha="center",
            va="bottom",
            clip_on=False,
        )

ax1.set_ylabel("WHP [PSI]")
ax1.legend(loc="upper left", fontsize="small")
ax1.grid(True)

# Bottom panel: DASCore waterfall only, with gap columns preserved as NaN.
ax2 = das_pump_patch.viz.waterfall(
    ax=ax2,
    cmap=das_cmap,
    scale=(-2, 2),
    scale_type="absolute",
    interpolation=None,
    cbar=True,
    show=False,
)
ax2.set_ylabel("Gold 4-PB Measured Depth [ft]")
ax2.set_xlabel("Time [UTC-7]")
ax2.set_ylim(plot_md_bottom, plot_md_top)
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d/%y\n%H:%M"))


# Add T1/T2/T3 reference times to both the pumping and DAS panels.
for mark_time, mark_label in zip(time_marks, mark_labels):
    if plot_start <= mark_time <= plot_end:
        for ax in (ax1, ax2):
            ax.axvline(mark_time, color="green", linestyle="--", linewidth=1.8, alpha=0.95)
        ax1.text(
            mark_time,
            -0.12,
            mark_label,
            color="green",
            fontsize=12,
            fontweight="bold",
            ha="center",
            va="top",
            transform=ax1.get_xaxis_transform(),
            clip_on=False,
        )

ax1.set_xlim(plot_start, plot_end)
ax2.set_xlim(plot_start, plot_end)

plt.show()




# Plot Strain waterfall map

In [ ]:
# Plot pumping curves on top of DASCore strain waterfall map
# Strain is computed from strain rate using DASCore Patch.integrate(dim="time").

plot_start = pd.to_datetime("2025-02-23 00:00")
plot_end = pd.to_datetime("2025-03-04 17:00")
plot_md_top = 9500
plot_md_bottom = 12250

# Time markers used to compare pumping stages with the DAS response.
time_marks = pd.to_datetime([
    "2025-02-24 11:00",
    "2025-02-28 00:00",
    "2025-03-03 22:00",
])
mark_labels = ["T1", "T2", "T3"]

Time_raw_dt = pd.DatetimeIndex(Time_raw)
depth_axis = np.linspace(lead, eor, LP_demean.shape[0])

time_mask = (Time_raw_dt >= plot_start) & (Time_raw_dt <= plot_end)
md_mask = (depth_axis >= plot_md_top) & (depth_axis <= plot_md_bottom)

Time_plot = Time_raw_dt[time_mask]
Depth_plot = depth_axis[md_mask]
LP_rate_plot = LP_demean[md_mask, :][:, time_mask]
gap_mask_plot = np.isnan(LP_rate_plot).all(axis=0)


def integrate_strain_rate_segments_with_dascore(rate_data, time_index, depth_values, gap_mask):
    """Integrate continuous non-gap strain-rate segments using DASCore."""
    rate_data = np.asarray(rate_data, dtype=float)
    time_index = pd.DatetimeIndex(time_index)
    strain_data = np.full_like(rate_data, np.nan, dtype=float)
    valid_time = ~gap_mask

    # Split into contiguous non-gap blocks so NaN gaps do not propagate through the cumulative strain.
    starts = np.flatnonzero(valid_time & np.r_[True, ~valid_time[:-1]])
    stops = np.flatnonzero(valid_time & np.r_[~valid_time[1:], True]) + 1

    for start_idx, stop_idx in zip(starts, stops):
        if stop_idx - start_idx < 2:
            continue

        segment_patch = dc.Patch(
            data=rate_data[:, start_idx:stop_idx],
            coords={
                "measured_depth": depth_values,
                "time": time_index[start_idx:stop_idx].to_numpy(dtype="datetime64[ns]"),
            },
            dims=("measured_depth", "time"),
            attrs={
                "data_type": "strain_rate",
                "data_units": "nanostrain/s",
                "time_zone": "UTC-7",
            },
        )
        strain_segment = segment_patch.integrate(dim="time", definite=False)
        strain_data[:, start_idx:stop_idx] = np.asarray(strain_segment.data)

    strain_data[:, gap_mask] = np.nan
    return strain_data


LP_strain_plot = integrate_strain_rate_segments_with_dascore(
    LP_rate_plot,
    Time_plot,
    Depth_plot,
    gap_mask_plot,
)

das_strain_patch = dc.Patch(
    data=LP_strain_plot / 1_000_000.0,  # nanostrain -> millistrain
    coords={
        "measured_depth": Depth_plot,
        "time": Time_plot.to_numpy(dtype="datetime64[ns]"),
    },
    dims=("measured_depth", "time"),
    attrs={
        "data_type": "strain",
        "data_units": "millistrain",
        "time_zone": "UTC-7",
        "time_gaps_filled_with": "NaN",
        "strain_conversion": "DASCore Patch.integrate(dim='time') on continuous non-gap segments",
    },
)

strain_cmap = plt.get_cmap("seismic").copy()
strain_cmap.set_bad("0.82")

fig, (ax1, ax2) = plt.subplots(
    2,
    1,
    figsize=(14, 6),
    sharex=True,
    gridspec_kw=dict(height_ratios=[1, 2.5]),
    constrained_layout=True,
)

color_map = {
    "Bearskin 1-IA WHP": "orange",
    "Bearskin 3-PA WHP": "red",
    "Bearskin 4-PB WHP": "purple",
}

for well, grp in whp_df.groupby("well"):
    ax1.plot(
        grp["time"],
        grp["pressure"],
        label=well,
        linewidth=1.5,
        color=color_map.get(well, "black"),
    )

stage_source = whp_df[(whp_df["well"] == "Bearskin 4-PB WHP") & whp_df["stage"].notna()].copy()
if not stage_source.empty:
    stage_source = stage_source.sort_values("time")
    stage_source["segment_id"] = (
        stage_source["stage"].ne(stage_source["stage"].shift())
        | stage_source["time"].diff().gt(pd.Timedelta(minutes=1.1))
    ).cumsum()
    stage_segments = (
        stage_source.groupby("segment_id")
        .agg(
            stage=("stage", "first"),
            start=("time", "min"),
            end=("time", "max"),
            ymax=("pressure", "max"),
        )
        .reset_index(drop=True)
    )
    pressure_pad = max(50, 0.03 * (stage_source["pressure"].max() - stage_source["pressure"].min()))
    for _, seg in stage_segments.iterrows():
        center_time = seg["start"] + (seg["end"] - seg["start"]) / 2
        ax1.text(
            center_time,
            seg["ymax"] + pressure_pad,
            f"{int(seg['stage'])}",
            color=color_map["Bearskin 4-PB WHP"],
            fontsize=9,
            fontweight="bold",
            ha="center",
            va="bottom",
            clip_on=False,
        )

ax1.set_ylabel("WHP [PSI]")
ax1.legend(loc="upper left", fontsize="small")
ax1.grid(True)

ax2 = das_strain_patch.viz.waterfall(
    ax=ax2,
    cmap=strain_cmap,
    scale=(-1, 1),
    scale_type="absolute",
    interpolation=None,
    cbar=True,
    show=False,
)
ax2.set_ylabel("Gold 4-PB Measured Depth [ft]")
ax2.set_xlabel("Time [UTC-7]")
ax2.set_ylim(plot_md_bottom, plot_md_top)
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d/%y\n%H:%M"))
if len(fig.axes) > 2:
    fig.axes[-1].set_ylabel("Strain (mε)")
ax2.set_title("DASCore Strain Waterfall Map Integrated from Strain Rate (mε)")


# Add T1/T2/T3 reference times to both the pumping and DAS panels.
for mark_time, mark_label in zip(time_marks, mark_labels):
    if plot_start <= mark_time <= plot_end:
        for ax in (ax1, ax2):
            ax.axvline(mark_time, color="green", linestyle="--", linewidth=1.8, alpha=0.95)
        ax1.text(
            mark_time,
            -0.12,
            mark_label,
            color="green",
            fontsize=12,
            fontweight="bold",
            ha="center",
            va="top",
            transform=ax1.get_xaxis_transform(),
            clip_on=False,
        )

ax1.set_xlim(plot_start, plot_end)
ax2.set_xlim(plot_start, plot_end)

plt.show()



In [ ]:
# DASCore strain-rate waterfall with overlaid 4-hour mean profiles and CSV export
# Target:
#   Plot the DASCore strain-rate waterfall plus WHP/pumping curves, then overlay one
#   4-hour mean strain-rate depth profile at the center of each 4-hour time window.
#   The exact plotted mean profile values are exported to CSV for reproducibility.

from pathlib import Path

plot_start = pd.to_datetime("2025-02-23 00:00")
plot_end = pd.to_datetime("2025-03-04 17:00")
plot_md_top = 9500
plot_md_bottom = 12250
profile_interval = "4h"
overlay_half_width_hours = 1.34

export_dir = Path(r"D:\OneDrive - Colorado School of Mines\Research\Fervo\Pengchao\code\profile_exports")
export_dir.mkdir(parents=True, exist_ok=True)
export_suffix = f"{plot_start:%Y%m%d_%H%M}_to_{plot_end:%Y%m%d_%H%M}_{int(plot_md_top)}_{int(plot_md_bottom)}ft_{profile_interval}_mean"

# T markers used in the pumping/waterfall figures.
time_marks = pd.to_datetime([
    "2025-02-24 11:00",
    "2025-02-28 00:00",
    "2025-03-03 22:00",
])
mark_labels = ["T1", "T2", "T3"]

Time_raw_dt = pd.DatetimeIndex(Time_raw)
depth_axis = np.linspace(lead, eor, LP_demean.shape[0])

time_mask = (Time_raw_dt >= plot_start) & (Time_raw_dt <= plot_end)
md_mask = (depth_axis >= plot_md_top) & (depth_axis <= plot_md_bottom)

Time_plot = Time_raw_dt[time_mask]
Depth_plot = depth_axis[md_mask]
LP_rate_plot = LP_demean[md_mask, :][:, time_mask]


def compute_mean_profiles(data, time_index, start_time, end_time, interval):
    """Compute one depth profile from the mean data in each regular time window."""
    time_index = pd.DatetimeIndex(time_index)
    window_starts = pd.date_range(start_time, end_time, freq=interval)
    profiles = []
    rows = []

    for window_start in window_starts:
        window_end = min(window_start + pd.Timedelta(interval), end_time)
        if window_start >= end_time:
            continue
        window_mask = (time_index >= window_start) & (time_index < window_end)
        if not window_mask.any():
            continue

        profile = np.nanmean(data[:, window_mask], axis=1)
        profiles.append(profile)
        rows.append(
            {
                "window_start": window_start,
                "window_end": window_end,
                "profile_center_time": window_start + (window_end - window_start) / 2,
                "n_samples": int(window_mask.sum()),
                "all_nan_profile": bool(np.isnan(profile).all()),
            }
        )

    profile_matrix = np.column_stack(profiles) if profiles else np.empty((data.shape[0], 0))
    profile_metadata = pd.DataFrame(rows)
    return profile_matrix, profile_metadata


def export_profile_csv(profile_matrix, profile_metadata, depth_values, data_csv, metadata_csv):
    """Export plotted mean profile values and their time-window metadata."""
    profile_df = pd.DataFrame({"measured_depth_ft": depth_values})
    for col_idx, row in profile_metadata.iterrows():
        col_name = pd.Timestamp(row["profile_center_time"]).strftime("%Y-%m-%d %H:%M:%S")
        profile_df[col_name] = profile_matrix[:, col_idx]
    profile_df.to_csv(data_csv, index=False)
    profile_metadata.to_csv(metadata_csv, index=False)
    return profile_df


def overlay_profiles_on_time_axis(ax, profile_matrix, profile_metadata, depth_values, half_width_hours, color="black"):
    """Overlay mean depth profiles on a time/depth waterfall by scaling amplitude into time offset."""
    if profile_matrix.size == 0:
        return np.nan

    profile_p95 = np.nanpercentile(np.abs(profile_matrix), 95)
    if not np.isfinite(profile_p95) or profile_p95 == 0:
        profile_p95 = np.nanmax(np.abs(profile_matrix))
    if not np.isfinite(profile_p95) or profile_p95 == 0:
        return np.nan

    scale_seconds_per_unit = half_width_hours * 3600.0 / profile_p95
    for col_idx, row in profile_metadata.iterrows():
        profile = profile_matrix[:, col_idx]
        finite = np.isfinite(profile)
        if not finite.any():
            continue
        center_time = pd.Timestamp(row["profile_center_time"])
        x_times = center_time + pd.to_timedelta(profile[finite] * scale_seconds_per_unit, unit="s")
        ax.plot(x_times, depth_values[finite], color=color, linewidth=0.8, alpha=0.9, zorder=5)

    return profile_p95

rate_profile_matrix, rate_profile_metadata = compute_mean_profiles(
    LP_rate_plot,
    Time_plot,
    plot_start,
    plot_end,
    profile_interval,
)
rate_profile_metadata["profile_units"] = "nanostrain/s"
rate_profile_metadata["overlay_half_width_hours"] = overlay_half_width_hours

rate_profile_csv = export_dir / f"strain_rate_4h_mean_profiles_{export_suffix}.csv"
rate_profile_meta_csv = export_dir / f"strain_rate_4h_mean_profiles_{export_suffix}_metadata.csv"
export_profile_csv(rate_profile_matrix, rate_profile_metadata, Depth_plot, rate_profile_csv, rate_profile_meta_csv)

rate_patch = dc.Patch(
    data=LP_rate_plot,
    coords={
        "measured_depth": Depth_plot,
        "time": Time_plot.to_numpy(dtype="datetime64[ns]"),
    },
    dims=("measured_depth", "time"),
    attrs={
        "data_type": "DAS strain rate",
        "data_units": "nanostrain/s",
        "time_zone": "UTC-7",
        "time_gaps_filled_with": "NaN",
    },
)

rate_cmap = plt.get_cmap("bwr").copy()
rate_cmap.set_bad("0.82")

fig, (ax1, ax2) = plt.subplots(
    2,
    1,
    figsize=(16, 8),
    sharex=True,
    gridspec_kw=dict(height_ratios=[1, 3]),
    constrained_layout=True,
)

color_map = {
    "Bearskin 1-IA WHP": "orange",
    "Bearskin 3-PA WHP": "red",
    "Bearskin 4-PB WHP": "purple",
}

pump_plot_df = whp_df[(whp_df["time"] >= plot_start) & (whp_df["time"] <= plot_end)].copy()
for well, grp in pump_plot_df.groupby("well"):
    ax1.plot(
        grp["time"],
        grp["pressure"],
        label=well,
        linewidth=1.5,
        color=color_map.get(well, "black"),
    )

# Label pumping stages on the Bearskin 4-PB WHP curve.
stage_source = pump_plot_df[(pump_plot_df["well"] == "Bearskin 4-PB WHP") & pump_plot_df["stage"].notna()].copy()
if not stage_source.empty:
    stage_source = stage_source.sort_values("time")
    stage_source["segment_id"] = (
        stage_source["stage"].ne(stage_source["stage"].shift())
        | stage_source["time"].diff().gt(pd.Timedelta(minutes=1.1))
    ).cumsum()
    stage_segments = (
        stage_source.groupby("segment_id")
        .agg(stage=("stage", "first"), start=("time", "min"), end=("time", "max"), ymax=("pressure", "max"))
        .reset_index(drop=True)
    )
    pressure_pad = max(50, 0.03 * (stage_source["pressure"].max() - stage_source["pressure"].min()))
    for _, seg in stage_segments.iterrows():
        center_time = seg["start"] + (seg["end"] - seg["start"]) / 2
        ax1.text(
            center_time,
            seg["ymax"] + pressure_pad,
            f"{int(seg['stage'])}",
            color=color_map["Bearskin 4-PB WHP"],
            fontsize=8,
            fontweight="bold",
            ha="center",
            va="bottom",
            clip_on=False,
        )

ax1.set_ylabel("WHP [PSI]")
ax1.legend(loc="upper left", fontsize="small")
ax1.grid(True)

ax2 = rate_patch.viz.waterfall(
    ax=ax2,
    cmap=rate_cmap,
    scale=(-2, 2),
    scale_type="absolute",
    interpolation=None,
    cbar=True,
    show=False,
)

profile_p95 = overlay_profiles_on_time_axis(
    ax2,
    rate_profile_matrix,
    rate_profile_metadata,
    Depth_plot,
    overlay_half_width_hours,
    color="black",
)
rate_profile_metadata["profile_p95_abs_for_overlay"] = profile_p95
rate_profile_metadata.to_csv(rate_profile_meta_csv, index=False)

ax2.set_ylabel("Gold 4-PB Measured Depth [ft]")
ax2.set_xlabel("Time [UTC-7]")
ax2.set_ylim(plot_md_bottom, plot_md_top)
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d/%y\n%H:%M"))
ax2.set_title("DASCore Strain-rate Waterfall with Overlaid 4-hour Mean Profiles")
ax2.text(
    0.01,
    0.02,
    f"Overlay scale: P95(|profile|)={profile_p95:.2f} n\u03b5/s maps to +/- {overlay_half_width_hours:.2f} hr",
    transform=ax2.transAxes,
    fontsize=9,
    bbox=dict(facecolor="white", edgecolor="0.6", alpha=0.85),
)

# Add T1/T2/T3 reference times to both time-based panels.
for mark_time, mark_label in zip(time_marks, mark_labels):
    if plot_start <= mark_time <= plot_end:
        for ax in (ax1, ax2):
            ax.axvline(mark_time, color="green", linestyle="--", linewidth=1.8, alpha=0.95)
        ax1.text(
            mark_time,
            -0.12,
            mark_label,
            color="green",
            fontsize=12,
            fontweight="bold",
            ha="center",
            va="top",
            transform=ax1.get_xaxis_transform(),
            clip_on=False,
        )

ax1.set_xlim(plot_start, plot_end)
ax2.set_xlim(plot_start, plot_end)

plt.show()

print(f"Plotted {rate_profile_matrix.shape[1]} 4-hour mean strain-rate profiles.")
print(f"Depth range: {plot_md_top}-{plot_md_bottom} ft; waterfall scale: -2 to 2 n\u03b5/s.")
print(f"Profile data CSV: {rate_profile_csv}")
print(f"Profile metadata CSV: {rate_profile_meta_csv}")


In [ ]:
# DASCore strain waterfall with overlaid 4-hour mean profiles and CSV export
# Target:
#   Integrate strain rate to strain using DASCore Patch.integrate(dim="time"), plot the
#   strain waterfall in millistrain, then overlay one 4-hour mean strain depth profile
#   at the center of each 4-hour time window. Exact plotted values are exported to CSV.

from pathlib import Path

plot_start = pd.to_datetime("2025-02-23 00:00")
plot_end = pd.to_datetime("2025-03-04 17:00")
plot_md_top = 9500
plot_md_bottom = 12250
profile_interval = "4h"
overlay_half_width_hours = 1.34

export_dir = Path(r"D:\OneDrive - Colorado School of Mines\Research\Fervo\Pengchao\code\profile_exports")
export_dir.mkdir(parents=True, exist_ok=True)
export_suffix = f"{plot_start:%Y%m%d_%H%M}_to_{plot_end:%Y%m%d_%H%M}_{int(plot_md_top)}_{int(plot_md_bottom)}ft_{profile_interval}_mean"

# T markers used in the pumping/waterfall figures.
time_marks = pd.to_datetime([
    "2025-02-24 11:00",
    "2025-02-28 00:00",
    "2025-03-03 22:00",
])
mark_labels = ["T1", "T2", "T3"]

Time_raw_dt = pd.DatetimeIndex(Time_raw)
depth_axis = np.linspace(lead, eor, LP_demean.shape[0])

time_mask = (Time_raw_dt >= plot_start) & (Time_raw_dt <= plot_end)
md_mask = (depth_axis >= plot_md_top) & (depth_axis <= plot_md_bottom)

Time_plot = Time_raw_dt[time_mask]
Depth_plot = depth_axis[md_mask]
LP_rate_plot = LP_demean[md_mask, :][:, time_mask]


def compute_mean_profiles(data, time_index, start_time, end_time, interval):
    """Compute one depth profile from the mean data in each regular time window."""
    time_index = pd.DatetimeIndex(time_index)
    window_starts = pd.date_range(start_time, end_time, freq=interval)
    profiles = []
    rows = []

    for window_start in window_starts:
        window_end = min(window_start + pd.Timedelta(interval), end_time)
        if window_start >= end_time:
            continue
        window_mask = (time_index >= window_start) & (time_index < window_end)
        if not window_mask.any():
            continue

        profile = np.nanmean(data[:, window_mask], axis=1)
        profiles.append(profile)
        rows.append(
            {
                "window_start": window_start,
                "window_end": window_end,
                "profile_center_time": window_start + (window_end - window_start) / 2,
                "n_samples": int(window_mask.sum()),
                "all_nan_profile": bool(np.isnan(profile).all()),
            }
        )

    profile_matrix = np.column_stack(profiles) if profiles else np.empty((data.shape[0], 0))
    profile_metadata = pd.DataFrame(rows)
    return profile_matrix, profile_metadata


def export_profile_csv(profile_matrix, profile_metadata, depth_values, data_csv, metadata_csv):
    """Export plotted mean profile values and their time-window metadata."""
    profile_df = pd.DataFrame({"measured_depth_ft": depth_values})
    for col_idx, row in profile_metadata.iterrows():
        col_name = pd.Timestamp(row["profile_center_time"]).strftime("%Y-%m-%d %H:%M:%S")
        profile_df[col_name] = profile_matrix[:, col_idx]
    profile_df.to_csv(data_csv, index=False)
    profile_metadata.to_csv(metadata_csv, index=False)
    return profile_df


def overlay_profiles_on_time_axis(ax, profile_matrix, profile_metadata, depth_values, half_width_hours, color="black"):
    """Overlay mean depth profiles on a time/depth waterfall by scaling amplitude into time offset."""
    if profile_matrix.size == 0:
        return np.nan

    profile_p95 = np.nanpercentile(np.abs(profile_matrix), 95)
    if not np.isfinite(profile_p95) or profile_p95 == 0:
        profile_p95 = np.nanmax(np.abs(profile_matrix))
    if not np.isfinite(profile_p95) or profile_p95 == 0:
        return np.nan

    scale_seconds_per_unit = half_width_hours * 3600.0 / profile_p95
    for col_idx, row in profile_metadata.iterrows():
        profile = profile_matrix[:, col_idx]
        finite = np.isfinite(profile)
        if not finite.any():
            continue
        center_time = pd.Timestamp(row["profile_center_time"])
        x_times = center_time + pd.to_timedelta(profile[finite] * scale_seconds_per_unit, unit="s")
        ax.plot(x_times, depth_values[finite], color=color, linewidth=0.8, alpha=0.9, zorder=5)

    return profile_p95

gap_mask_plot = np.isnan(LP_rate_plot).all(axis=0)


def integrate_strain_rate_segments_with_dascore(rate_data, time_index, depth_values, gap_mask):
    """Integrate continuous non-gap strain-rate segments using DASCore."""
    rate_data = np.asarray(rate_data, dtype=float)
    time_index = pd.DatetimeIndex(time_index)
    strain_data = np.full_like(rate_data, np.nan, dtype=float)
    valid_time = ~gap_mask

    # Split into contiguous non-gap blocks so missing timeline gaps are not bridged by integration.
    starts = np.flatnonzero(valid_time & np.r_[True, ~valid_time[:-1]])
    stops = np.flatnonzero(valid_time & np.r_[~valid_time[1:], True]) + 1

    for start_idx, stop_idx in zip(starts, stops):
        if stop_idx - start_idx < 2:
            continue

        segment_patch = dc.Patch(
            data=rate_data[:, start_idx:stop_idx],
            coords={
                "measured_depth": depth_values,
                "time": time_index[start_idx:stop_idx].to_numpy(dtype="datetime64[ns]"),
            },
            dims=("measured_depth", "time"),
            attrs={
                "data_type": "strain_rate",
                "data_units": "nanostrain/s",
                "time_zone": "UTC-7",
            },
        )
        strain_segment = segment_patch.integrate(dim="time", definite=False)
        strain_data[:, start_idx:stop_idx] = np.asarray(strain_segment.data)

    strain_data[:, gap_mask] = np.nan
    return strain_data


LP_strain_plot = integrate_strain_rate_segments_with_dascore(
    LP_rate_plot,
    Time_plot,
    Depth_plot,
    gap_mask_plot,
)
LP_strain_mstrain = LP_strain_plot / 1_000_000.0

strain_profile_matrix, strain_profile_metadata = compute_mean_profiles(
    LP_strain_mstrain,
    Time_plot,
    plot_start,
    plot_end,
    profile_interval,
)
strain_profile_metadata["profile_units"] = "millistrain"
strain_profile_metadata["overlay_half_width_hours"] = overlay_half_width_hours

strain_profile_csv = export_dir / f"strain_4h_mean_profiles_{export_suffix}.csv"
strain_profile_meta_csv = export_dir / f"strain_4h_mean_profiles_{export_suffix}_metadata.csv"
export_profile_csv(strain_profile_matrix, strain_profile_metadata, Depth_plot, strain_profile_csv, strain_profile_meta_csv)

strain_patch = dc.Patch(
    data=LP_strain_mstrain,
    coords={
        "measured_depth": Depth_plot,
        "time": Time_plot.to_numpy(dtype="datetime64[ns]"),
    },
    dims=("measured_depth", "time"),
    attrs={
        "data_type": "strain",
        "data_units": "millistrain",
        "time_zone": "UTC-7",
        "time_gaps_filled_with": "NaN",
        "strain_conversion": "DASCore Patch.integrate(dim='time') on continuous non-gap segments",
    },
)

strain_cmap = plt.get_cmap("seismic").copy()
strain_cmap.set_bad("0.82")

fig, (ax1, ax2) = plt.subplots(
    2,
    1,
    figsize=(16, 8),
    sharex=True,
    gridspec_kw=dict(height_ratios=[1, 3]),
    constrained_layout=True,
)

color_map = {
    "Bearskin 1-IA WHP": "orange",
    "Bearskin 3-PA WHP": "red",
    "Bearskin 4-PB WHP": "purple",
}

pump_plot_df = whp_df[(whp_df["time"] >= plot_start) & (whp_df["time"] <= plot_end)].copy()
for well, grp in pump_plot_df.groupby("well"):
    ax1.plot(
        grp["time"],
        grp["pressure"],
        label=well,
        linewidth=1.5,
        color=color_map.get(well, "black"),
    )

# Label pumping stages on the Bearskin 4-PB WHP curve.
stage_source = pump_plot_df[(pump_plot_df["well"] == "Bearskin 4-PB WHP") & pump_plot_df["stage"].notna()].copy()
if not stage_source.empty:
    stage_source = stage_source.sort_values("time")
    stage_source["segment_id"] = (
        stage_source["stage"].ne(stage_source["stage"].shift())
        | stage_source["time"].diff().gt(pd.Timedelta(minutes=1.1))
    ).cumsum()
    stage_segments = (
        stage_source.groupby("segment_id")
        .agg(stage=("stage", "first"), start=("time", "min"), end=("time", "max"), ymax=("pressure", "max"))
        .reset_index(drop=True)
    )
    pressure_pad = max(50, 0.03 * (stage_source["pressure"].max() - stage_source["pressure"].min()))
    for _, seg in stage_segments.iterrows():
        center_time = seg["start"] + (seg["end"] - seg["start"]) / 2
        ax1.text(
            center_time,
            seg["ymax"] + pressure_pad,
            f"{int(seg['stage'])}",
            color=color_map["Bearskin 4-PB WHP"],
            fontsize=8,
            fontweight="bold",
            ha="center",
            va="bottom",
            clip_on=False,
        )

ax1.set_ylabel("WHP [PSI]")
ax1.legend(loc="upper left", fontsize="small")
ax1.grid(True)

ax2 = strain_patch.viz.waterfall(
    ax=ax2,
    cmap=strain_cmap,
    scale=(-1, 1),
    scale_type="absolute",
    interpolation=None,
    cbar=True,
    show=False,
)
if len(fig.axes) > 2:
    fig.axes[-1].set_ylabel("Strain (m\u03b5)")

profile_p95 = overlay_profiles_on_time_axis(
    ax2,
    strain_profile_matrix,
    strain_profile_metadata,
    Depth_plot,
    overlay_half_width_hours,
    color="black",
)
strain_profile_metadata["profile_p95_abs_for_overlay"] = profile_p95
strain_profile_metadata.to_csv(strain_profile_meta_csv, index=False)

ax2.set_ylabel("Gold 4-PB Measured Depth [ft]")
ax2.set_xlabel("Time [UTC-7]")
ax2.set_ylim(plot_md_bottom, plot_md_top)
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d/%y\n%H:%M"))
ax2.set_title("DASCore Strain Waterfall with Overlaid 4-hour Mean Profiles (m\u03b5)")
ax2.text(
    0.01,
    0.02,
    f"Overlay scale: P95(|profile|)={profile_p95:.2f} m\u03b5 maps to +/- {overlay_half_width_hours:.2f} hr",
    transform=ax2.transAxes,
    fontsize=9,
    bbox=dict(facecolor="white", edgecolor="0.6", alpha=0.85),
)

# Add T1/T2/T3 reference times to both time-based panels.
for mark_time, mark_label in zip(time_marks, mark_labels):
    if plot_start <= mark_time <= plot_end:
        for ax in (ax1, ax2):
            ax.axvline(mark_time, color="green", linestyle="--", linewidth=1.8, alpha=0.95)
        ax1.text(
            mark_time,
            -0.12,
            mark_label,
            color="green",
            fontsize=12,
            fontweight="bold",
            ha="center",
            va="top",
            transform=ax1.get_xaxis_transform(),
            clip_on=False,
        )

ax1.set_xlim(plot_start, plot_end)
ax2.set_xlim(plot_start, plot_end)

plt.show()

print(f"Plotted {strain_profile_matrix.shape[1]} 4-hour mean strain profiles.")
print(f"Depth range: {plot_md_top}-{plot_md_bottom} ft; waterfall scale: -1 to 1 m\u03b5.")
print(f"Profile data CSV: {strain_profile_csv}")
print(f"Profile metadata CSV: {strain_profile_meta_csv}")


# Observation within V1 mds range

In [ ]:
# T1-to-T3 strain-rate and T1-referenced strain waterfalls with 4-hour mean profile overlays
# Requested view:
#   - depth limited to 10200-10500 ft
#   - time limited to T1-T3
#   - no pumping or injection curves
#   - show strain rate first, then strain
#   - use T1 as the strain reference so the T1 strain profile is zero
#   - overlay profiles every 4 hours, consistent with the previous cells
#   - export strain-rate and strain profile values for future inversion

from pathlib import Path

plot_start = pd.to_datetime("2025-02-24 11:00")  # T1
plot_end = pd.to_datetime("2025-03-03 22:00")    # T3
plot_md_top = 10200
plot_md_bottom = 10500
profile_interval = "4h"
rate_overlay_half_width_hours = 28.0
strain_overlay_half_width_hours = 28.0
profile_scale_multiplier = 10.0  # Keep the overlay scale comparable to the pre-GL-corrected plots.
rate_color_scale = (-0.3, 0.3)      # nanostrain/s
strain_color_scale = (-0.1, 0.1)  # millistrain

export_dir = Path(r"D:\OneDrive - Colorado School of Mines\Research\Fervo\Pengchao\code\profile_exports")
export_dir.mkdir(parents=True, exist_ok=True)
export_suffix = (
    f"{plot_start:%Y%m%d_%H%M}_to_{plot_end:%Y%m%d_%H%M}_"
    f"{int(plot_md_top)}_{int(plot_md_bottom)}ft_{profile_interval}_mean_T1_ref"
)

# T markers used in the figure.
time_marks = pd.to_datetime([
    "2025-02-24 11:00",
    "2025-02-28 00:00",
    "2025-03-03 22:00",
])
mark_labels = ["T1", "T2", "T3"]
reference_profile_time = pd.to_datetime("2025-03-03 00:00")

Time_raw_dt = pd.DatetimeIndex(Time_raw)
depth_axis = np.linspace(lead, eor, LP_demean.shape[0])

time_mask = (Time_raw_dt >= plot_start) & (Time_raw_dt <= plot_end)
md_mask = (depth_axis >= plot_md_top) & (depth_axis <= plot_md_bottom)

Time_plot = Time_raw_dt[time_mask]
Depth_plot = depth_axis[md_mask]
LP_rate_plot = LP_demean[md_mask, :][:, time_mask]
gap_mask_plot = np.isnan(LP_rate_plot).all(axis=0)


def compute_mean_profiles(data, time_index, start_time, end_time, interval):
    """Compute one depth profile from the mean data in each regular time window."""
    time_index = pd.DatetimeIndex(time_index)
    window_starts = pd.date_range(start_time, end_time, freq=interval)
    profiles = []
    rows = []

    for window_start in window_starts:
        window_end = min(window_start + pd.Timedelta(interval), end_time)
        if window_start >= end_time:
            continue
        window_mask = (time_index >= window_start) & (time_index < window_end)
        if not window_mask.any():
            continue

        profile = np.nanmean(data[:, window_mask], axis=1)
        profiles.append(profile)
        rows.append(
            {
                "window_start": window_start,
                "window_end": window_end,
                "profile_center_time": window_start,  # Anchor each plotted profile at its window start (first = T1).
                "n_samples": int(window_mask.sum()),
                "all_nan_profile": bool(np.isnan(profile).all()),
            }
        )

    profile_matrix = np.column_stack(profiles) if profiles else np.empty((data.shape[0], 0))
    profile_metadata = pd.DataFrame(rows)
    return profile_matrix, profile_metadata


def export_profile_csv(profile_matrix, profile_metadata, depth_values, data_csv, metadata_csv):
    """Export profile values exactly as plotted, plus time-window metadata."""
    profile_df = pd.DataFrame({"measured_depth_ft": depth_values})
    for col_idx, row in profile_metadata.iterrows():
        col_name = pd.Timestamp(row["profile_center_time"]).strftime("%Y-%m-%d %H:%M:%S")
        profile_df[col_name] = profile_matrix[:, col_idx]
    profile_df.to_csv(data_csv, index=False)
    profile_metadata.to_csv(metadata_csv, index=False)
    return profile_df


def get_profile_abs_scale(profile_matrix, multiplier=1.0):
    """Return a robust absolute scale used to convert profile amplitude into time width."""
    if profile_matrix.size == 0:
        return np.nan, np.nan
    profile_p95 = np.nanpercentile(np.abs(profile_matrix), 95)
    if not np.isfinite(profile_p95) or profile_p95 == 0:
        profile_p95 = np.nanmax(np.abs(profile_matrix))
    if not np.isfinite(profile_p95) or profile_p95 == 0:
        return np.nan, np.nan
    return profile_p95, profile_p95 * multiplier


def overlay_profiles_on_time_axis(
    ax,
    profile_matrix,
    profile_metadata,
    depth_values,
    half_width_hours,
    scale_reference_abs,
    color="black",
):
    """Overlay mean depth profiles on a time/depth waterfall by scaling amplitude into time offset."""
    if profile_matrix.size == 0 or not np.isfinite(scale_reference_abs) or scale_reference_abs == 0:
        return

    scale_seconds_per_unit = half_width_hours * 3600.0 / scale_reference_abs
    for col_idx, row in profile_metadata.iterrows():
        profile = profile_matrix[:, col_idx]
        finite = np.isfinite(profile)
        if not finite.any():
            continue
        center_time = pd.Timestamp(row["profile_center_time"])
        x_times = center_time + pd.to_timedelta(profile[finite] * scale_seconds_per_unit, unit="s")
        ax.plot(x_times, depth_values[finite], color=color, linewidth=0.8, alpha=0.9, zorder=5)


def annotate_reference_profile_peak(
    ax,
    profile_matrix,
    profile_metadata,
    depth_values,
    half_width_hours,
    scale_reference_abs,
    target_time,
    unit_label,
    label_prefix,
):
    """Mark the peak absolute amplitude of the 4-hour profile nearest target_time."""
    if profile_matrix.size == 0 or profile_metadata.empty:
        return None
    if not np.isfinite(scale_reference_abs) or scale_reference_abs == 0:
        return None

    centers = pd.to_datetime(profile_metadata["profile_center_time"])
    target_time = pd.Timestamp(target_time)
    col_idx = int(np.argmin(np.abs(centers - target_time)))
    profile = profile_matrix[:, col_idx]
    finite = np.isfinite(profile)
    if not finite.any():
        return None

    finite_indices = np.flatnonzero(finite)
    peak_local_idx = int(np.nanargmax(np.abs(profile[finite])))
    peak_idx = int(finite_indices[peak_local_idx])
    peak_value = float(profile[peak_idx])
    peak_depth = float(depth_values[peak_idx])
    center_time = pd.Timestamp(centers.iloc[col_idx])

    scale_seconds_per_unit = half_width_hours * 3600.0 / scale_reference_abs
    peak_time = center_time + pd.to_timedelta(peak_value * scale_seconds_per_unit, unit="s")

    ax.scatter(
        [peak_time],
        [peak_depth],
        marker="*",
        color="yellow",
        edgecolor="black",
        linewidth=0.8,
        s=150,
        zorder=10,
    )
    ax.text(
        peak_time,
        peak_depth + 8,
        f"{peak_value:+.3g} {unit_label}",
        color="black",
        fontsize=9,
        ha="center",
        va="top",
        bbox=dict(facecolor="white", edgecolor="black", alpha=0.88, pad=2),
        zorder=11,
    )
    return {
        "center_time": center_time,
        "peak_value": peak_value,
        "peak_depth": peak_depth,
        "target_time": target_time,
    }

def integrate_strain_rate_segments_with_dascore(rate_data, time_index, depth_values, gap_mask):
    """Integrate continuous non-gap strain-rate segments using DASCore."""
    rate_data = np.asarray(rate_data, dtype=float)
    time_index = pd.DatetimeIndex(time_index)
    strain_data = np.full_like(rate_data, np.nan, dtype=float)
    valid_time = ~gap_mask

    starts = np.flatnonzero(valid_time & np.r_[True, ~valid_time[:-1]])
    stops = np.flatnonzero(valid_time & np.r_[~valid_time[1:], True]) + 1

    for start_idx, stop_idx in zip(starts, stops):
        if stop_idx - start_idx < 2:
            continue

        segment_patch = dc.Patch(
            data=rate_data[:, start_idx:stop_idx],
            coords={
                "measured_depth": depth_values,
                "time": time_index[start_idx:stop_idx].to_numpy(dtype="datetime64[ns]"),
            },
            dims=("measured_depth", "time"),
            attrs={
                "data_type": "strain_rate",
                "data_units": "nanostrain/s",
                "time_zone": "UTC-7",
            },
        )
        strain_segment = segment_patch.integrate(dim="time", definite=False)
        strain_data[:, start_idx:stop_idx] = np.asarray(strain_segment.data)

    strain_data[:, gap_mask] = np.nan
    return strain_data


LP_strain_plot = integrate_strain_rate_segments_with_dascore(
    LP_rate_plot,
    Time_plot,
    Depth_plot,
    gap_mask_plot,
)
LP_strain_mstrain = LP_strain_plot / 1_000_000.0

rate_profile_matrix, rate_profile_metadata = compute_mean_profiles(
    LP_rate_plot,
    Time_plot,
    plot_start,
    plot_end,
    profile_interval,
)
rate_profile_metadata["profile_units"] = "nanostrain/s"
rate_profile_metadata["overlay_half_width_hours"] = rate_overlay_half_width_hours
rate_profile_metadata["overlay_scale_multiplier"] = profile_scale_multiplier

# Use the first 4-hour window starting at T1 as the strain reference for profile display.
strain_profile_matrix_raw, strain_profile_metadata = compute_mean_profiles(
    LP_strain_mstrain,
    Time_plot,
    plot_start,
    plot_end,
    profile_interval,
)

if strain_profile_matrix_raw.shape[1] > 0:
    strain_reference_profile = strain_profile_matrix_raw[:, [0]]
else:
    strain_reference_profile = np.zeros((len(Depth_plot), 1))

strain_profile_matrix = strain_profile_matrix_raw - strain_reference_profile
strain_profile_metadata["profile_units"] = "millistrain"
strain_profile_metadata["overlay_half_width_hours"] = strain_overlay_half_width_hours
strain_profile_metadata["overlay_scale_multiplier"] = profile_scale_multiplier

# Reference the strain waterfall to the first 4-hour T1 window so the T1 strain profile is zero.
LP_strain_ref_mstrain = LP_strain_mstrain - strain_reference_profile

rate_profile_p95, rate_profile_scale_reference = get_profile_abs_scale(
    rate_profile_matrix,
    multiplier=profile_scale_multiplier,
)
strain_profile_p95, strain_profile_scale_reference = get_profile_abs_scale(
    strain_profile_matrix,
    multiplier=profile_scale_multiplier,
)
rate_profile_metadata["profile_p95_abs"] = rate_profile_p95
rate_profile_metadata["overlay_scale_reference_abs"] = rate_profile_scale_reference
strain_profile_metadata["profile_p95_abs"] = strain_profile_p95
strain_profile_metadata["overlay_scale_reference_abs"] = strain_profile_scale_reference

rate_profile_csv = export_dir / f"strain_rate_4h_mean_profiles_{export_suffix}.csv"
rate_profile_meta_csv = export_dir / f"strain_rate_4h_mean_profiles_{export_suffix}_metadata.csv"
strain_profile_csv = export_dir / f"strain_4h_mean_profiles_{export_suffix}.csv"
strain_profile_meta_csv = export_dir / f"strain_4h_mean_profiles_{export_suffix}_metadata.csv"
export_profile_csv(rate_profile_matrix, rate_profile_metadata, Depth_plot, rate_profile_csv, rate_profile_meta_csv)
export_profile_csv(strain_profile_matrix, strain_profile_metadata, Depth_plot, strain_profile_csv, strain_profile_meta_csv)

rate_patch = dc.Patch(
    data=LP_rate_plot,
    coords={
        "measured_depth": Depth_plot,
        "time": Time_plot.to_numpy(dtype="datetime64[ns]"),
    },
    dims=("measured_depth", "time"),
    attrs={
        "data_type": "DAS strain rate",
        "data_units": "nanostrain/s",
        "time_zone": "UTC-7",
        "time_gaps_filled_with": "NaN",
    },
)

strain_patch = dc.Patch(
    data=LP_strain_ref_mstrain,
    coords={
        "measured_depth": Depth_plot,
        "time": Time_plot.to_numpy(dtype="datetime64[ns]"),
    },
    dims=("measured_depth", "time"),
    attrs={
        "data_type": "strain",
        "data_units": "millistrain",
        "time_zone": "UTC-7",
        "time_gaps_filled_with": "NaN",
        "strain_reference": f"first {profile_interval} window starting at T1 = {plot_start}",
    },
)

rate_cmap = plt.get_cmap("bwr").copy()
rate_cmap.set_bad("0.82")
strain_cmap = plt.get_cmap("seismic").copy()
strain_cmap.set_bad("0.82")

fig, (ax1, ax2) = plt.subplots(
    2,
    1,
    figsize=(16, 9),
    sharex=True,
    constrained_layout=True,
)

ax1 = rate_patch.viz.waterfall(
    ax=ax1,
    cmap=rate_cmap,
    scale=rate_color_scale,
    scale_type="absolute",
    interpolation=None,
    cbar=True,
    show=False,
)
if len(fig.axes) > 2:
    fig.axes[2].set_ylabel("Strain rate (nanostrain/s)")
overlay_profiles_on_time_axis(
    ax1,
    rate_profile_matrix,
    rate_profile_metadata,
    Depth_plot,
    rate_overlay_half_width_hours,
    rate_profile_scale_reference,
    color="black",
)
ax1.set_ylabel("Gold 4-PB Measured Depth [ft]")
ax1.set_ylim(plot_md_bottom, plot_md_top)
ax1.set_title("DASCore Strain-rate Waterfall with Overlaid 4-hour Mean Profiles")

ax2 = strain_patch.viz.waterfall(
    ax=ax2,
    cmap=strain_cmap,
    scale=strain_color_scale,
    scale_type="absolute",
    interpolation=None,
    cbar=True,
    show=False,
)
if len(fig.axes) > 3:
    fig.axes[3].set_ylabel("Strain (millistrain)")
overlay_profiles_on_time_axis(
    ax2,
    strain_profile_matrix,
    strain_profile_metadata,
    Depth_plot,
    strain_overlay_half_width_hours,
    strain_profile_scale_reference,
    color="black",
)
ax2.set_ylabel("Gold 4-PB Measured Depth [ft]")
ax2.set_xlabel("Time [UTC-7]")
ax2.set_ylim(plot_md_bottom, plot_md_top)
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d\n%H:%M"))
ax2.set_title("DASCore Strain Waterfall with Overlaid 4-hour Mean Profiles (T1 Referenced)")

# Mark T1/T2/T3 with green dashed reference lines on both panels.
for mark_time, mark_label in zip(time_marks, mark_labels):
    for ax in (ax1, ax2):
        ax.axvline(mark_time, color="green", linestyle="--", linewidth=1.8, alpha=0.95, zorder=4)
        ax.text(
            mark_time,
            1.01,
            mark_label,
            color="green",
            fontsize=12,
            fontweight="bold",
            ha="center",
            va="bottom",
            transform=ax.get_xaxis_transform(),
            clip_on=False,
        )

rate_reference_info = annotate_reference_profile_peak(
    ax1,
    rate_profile_matrix,
    rate_profile_metadata,
    Depth_plot,
    rate_overlay_half_width_hours,
    rate_profile_scale_reference,
    reference_profile_time,
    "nanostrain/s",
    "Strain-rate",
)
strain_reference_info = annotate_reference_profile_peak(
    ax2,
    strain_profile_matrix,
    strain_profile_metadata,
    Depth_plot,
    strain_overlay_half_width_hours,
    strain_profile_scale_reference,
    reference_profile_time,
    "millistrain",
    "Strain",
)

ax1.set_xlim(plot_start, plot_end)
ax2.set_xlim(plot_start, plot_end)

plt.show()

print("Created T1-to-T3 strain-rate and T1-referenced strain waterfall views with 4-hour profile overlays.")
print(f"Depth range: {plot_md_top}-{plot_md_bottom} ft; time range: {plot_start} to {plot_end}.")
print(f"Color scales: strain rate {rate_color_scale} nanostrain/s; strain {strain_color_scale} millistrain.")
print(f"Reference profile target time: {reference_profile_time}.")
if rate_reference_info is not None:
    print(f"Strain-rate reference profile center: {rate_reference_info['center_time']}; peak: {rate_reference_info['peak_value']:+.6g} nanostrain/s at {rate_reference_info['peak_depth']:.1f} ft.")
if strain_reference_info is not None:
    print(f"Strain reference profile center: {strain_reference_info['center_time']}; peak: {strain_reference_info['peak_value']:+.6g} millistrain at {strain_reference_info['peak_depth']:.1f} ft.")
print(f"Strain reference profile uses the first {profile_interval} window starting at T1.")
print(f"Plotted/exported {rate_profile_matrix.shape[1]} strain-rate profiles and {strain_profile_matrix.shape[1]} strain profiles.")
print(f"Strain-rate profile CSV: {rate_profile_csv}")
print(f"Strain-rate metadata CSV: {rate_profile_meta_csv}")
print(f"Strain profile CSV: {strain_profile_csv}")
print(f"Strain metadata CSV: {strain_profile_meta_csv}")


# Simulation within V1 mds range

In [ ]:
# Direct-DDM test-fault-length-2 waterfall/profile view from sim_fault2 CSV exports
# Run the modeling notebook's simulated export cell first. This cell reads those CSVs and
# plots simulated strain-rate and T1-referenced strain using the same 4-hour profile convention.

from pathlib import Path
import matplotlib.dates as mdates

sim_export_dir = Path(r"D:\OneDrive - Colorado School of Mines\Research\Fervo\Pengchao\code\profile_exports\sim_fault2")
sim_profile_interval = "4h"
sim_rate_overlay_half_width_hours = 28.0
sim_strain_overlay_half_width_hours = 28.0
sim_profile_scale_multiplier = 10.0
sim_rate_color_scale = (-0.3, 0.3)      # nanostrain/s
sim_strain_color_scale = (-0.1, 0.1)    # millistrain
sim_plot_start = pd.to_datetime("2025-02-24 11:00")
sim_plot_end = pd.to_datetime("2025-03-03 22:00")
sim_md_top = 10200
sim_md_bottom = 10500
sim_time_marks = pd.to_datetime([
    "2025-02-24 11:00",
    "2025-02-28 00:00",
    "2025-03-03 22:00",
])
sim_mark_labels = ["T1", "T2", "T3"]
reference_profile_time = pd.to_datetime("2025-03-03 00:00")


def latest_file(pattern, exclude_metadata=False):
    files = list(sim_export_dir.glob(pattern))
    if exclude_metadata:
        files = [path for path in files if not path.name.endswith("_metadata.csv")]
    if not files:
        raise FileNotFoundError(
            f"No file matching {pattern!r} found in {sim_export_dir}. "
            "Run the simulated export cell in the modeling notebook first."
        )
    return max(files, key=lambda p: p.stat().st_mtime)


sim_rate_waterfall_csv = latest_file("sim_rate_waterfall_*.csv")
sim_strain_waterfall_csv = latest_file("sim_direct_strain_waterfall_*.csv")
sim_rate_profile_csv = latest_file("sim_rate_4h_*.csv", exclude_metadata=True)
sim_rate_profile_meta_csv = latest_file("sim_rate_4h_*_metadata.csv")
sim_strain_profile_csv = latest_file("sim_direct_strain_4h_*.csv", exclude_metadata=True)
sim_strain_profile_meta_csv = latest_file("sim_direct_strain_4h_*_metadata.csv")


def read_depth_time_csv(csv_path):
    df = pd.read_csv(csv_path)
    depth = df["measured_depth_ft"].to_numpy(dtype=float)
    time = pd.DatetimeIndex(pd.to_datetime(df.columns[1:]))
    data = df.iloc[:, 1:].to_numpy(dtype=float)
    return data, depth, time


def read_profile_csv(data_csv, metadata_csv):
    profile_df = pd.read_csv(data_csv)
    metadata = pd.read_csv(metadata_csv)
    depth = profile_df["measured_depth_ft"].to_numpy(dtype=float)
    profiles = profile_df.iloc[:, 1:].to_numpy(dtype=float)
    metadata["window_start"] = pd.to_datetime(metadata["window_start"])
    metadata["window_end"] = pd.to_datetime(metadata["window_end"])
    metadata["profile_center_time"] = pd.to_datetime(metadata["profile_center_time"])
    return profiles, metadata, depth


def get_profile_abs_scale(profile_matrix, multiplier=1.0):
    if profile_matrix.size == 0:
        return np.nan, np.nan
    profile_p95 = np.nanpercentile(np.abs(profile_matrix), 95)
    if not np.isfinite(profile_p95) or profile_p95 == 0:
        profile_p95 = np.nanmax(np.abs(profile_matrix))
    if not np.isfinite(profile_p95) or profile_p95 == 0:
        return np.nan, np.nan
    return profile_p95, profile_p95 * multiplier


def overlay_profiles_on_time_axis(ax, profile_matrix, profile_metadata, depth_values, half_width_hours, scale_reference_abs, color="black"):
    if profile_matrix.size == 0 or not np.isfinite(scale_reference_abs) or scale_reference_abs == 0:
        return
    scale_seconds_per_unit = half_width_hours * 3600.0 / scale_reference_abs
    for col_idx, row in profile_metadata.iterrows():
        profile = profile_matrix[:, col_idx]
        finite = np.isfinite(profile)
        if not finite.any():
            continue
        center_time = pd.Timestamp(row["profile_center_time"])
        x_times = center_time + pd.to_timedelta(profile[finite] * scale_seconds_per_unit, unit="s")
        ax.plot(x_times, depth_values[finite], color=color, linewidth=0.8, alpha=0.9, zorder=5)


def annotate_reference_profile_peak(
    ax,
    profile_matrix,
    profile_metadata,
    profile_depth,
    overlay_half_width_hours,
    scale_reference_abs,
    target_time,
    unit_label,
    profile_label,
):
    if profile_matrix.size == 0 or profile_metadata.empty:
        return None
    profile_times = pd.to_datetime(profile_metadata["profile_center_time"])
    profile_idx = int(np.argmin(np.abs(profile_times - target_time)))
    profile_values = profile_matrix[:, profile_idx]
    if profile_values.size == 0 or np.all(~np.isfinite(profile_values)):
        return None
    peak_idx = int(np.nanargmax(np.abs(profile_values)))
    peak_value = float(profile_values[peak_idx])
    peak_depth = float(profile_depth[peak_idx])
    peak_time = profile_times.iloc[profile_idx] + pd.to_timedelta(
        overlay_half_width_hours * 3600.0 * peak_value / scale_reference_abs,
        unit="s",
    )
    ax.scatter(
        peak_time,
        peak_depth,
        marker="*",
        color="yellow",
        edgecolor="black",
        linewidth=0.8,
        s=150,
        zorder=10,
    )
    ax.text(
        peak_time,
        peak_depth + 8,
        f"{peak_value:+.3g} {unit_label}",
        color="black",
        fontsize=9,
        fontweight="bold",
        ha="center",
        va="top",
        bbox=dict(facecolor="yellow", edgecolor="black", alpha=0.85, boxstyle="round,pad=0.2"),
        zorder=11,
    )
    return {
        "profile_label": profile_label,
        "profile_time": profile_times.iloc[profile_idx],
        "target_time": target_time,
        "peak_depth_ft": peak_depth,
        "peak_value": peak_value,
        "unit": unit_label,
    }


def plot_sim_waterfall(ax, data_matrix, time_values, depth_values, cmap_name, color_scale, cbar_label, title):
    x0 = mdates.date2num(pd.Timestamp(time_values[0]).to_pydatetime())
    x1 = mdates.date2num(pd.Timestamp(time_values[-1]).to_pydatetime())
    im = ax.imshow(
        data_matrix,
        cmap=cmap_name,
        aspect="auto",
        vmin=color_scale[0],
        vmax=color_scale[1],
        extent=(x0, x1, depth_values[-1], depth_values[0]),
        interpolation="none",
    )
    ax.xaxis_date()
    ax.set_ylabel("Measured Depth (ft)")
    ax.set_ylim(sim_md_bottom, sim_md_top)
    ax.set_title(title)
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label(cbar_label)
    return im


SimRate_plot, SimDepth_rate, SimTime_plot = read_depth_time_csv(sim_rate_waterfall_csv)
SimStrain_plot, SimDepth_strain, SimStrainTime_plot = read_depth_time_csv(sim_strain_waterfall_csv)
SimRate_profile_matrix, sim_rate_profile_metadata, SimProfileDepth_rate = read_profile_csv(sim_rate_profile_csv, sim_rate_profile_meta_csv)
SimStrain_profile_matrix, sim_strain_profile_metadata, SimProfileDepth_strain = read_profile_csv(sim_strain_profile_csv, sim_strain_profile_meta_csv)

sim_rate_p95, sim_rate_scale_reference = get_profile_abs_scale(SimRate_profile_matrix, sim_profile_scale_multiplier)
sim_strain_p95, sim_strain_scale_reference = get_profile_abs_scale(SimStrain_profile_matrix, sim_profile_scale_multiplier)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 9), sharex=True, constrained_layout=True)
plot_sim_waterfall(
    ax1,
    SimRate_plot,
    SimTime_plot,
    SimDepth_rate,
    "bwr",
    sim_rate_color_scale,
    "Strain rate (nanostrain/s)",
    "Simulated strain-rate waterfall with overlaid 4-hour mean profiles",
)
overlay_profiles_on_time_axis(
    ax1,
    SimRate_profile_matrix,
    sim_rate_profile_metadata,
    SimProfileDepth_rate,
    sim_rate_overlay_half_width_hours,
    sim_rate_scale_reference,
)
plot_sim_waterfall(
    ax2,
    SimStrain_plot,
    SimStrainTime_plot,
    SimDepth_strain,
    "seismic",
    sim_strain_color_scale,
    "Strain (millistrain)",
    "Direct DDM strain waterfall with overlaid 4-hour mean profiles (T1 referenced)",
)
overlay_profiles_on_time_axis(
    ax2,
    SimStrain_profile_matrix,
    sim_strain_profile_metadata,
    SimProfileDepth_strain,
    sim_strain_overlay_half_width_hours,
    sim_strain_scale_reference,
)

for mark_time, mark_label in zip(sim_time_marks, sim_mark_labels):
    for ax in (ax1, ax2):
        ax.axvline(mark_time, color="green", linestyle="--", linewidth=1.8, alpha=0.95, zorder=4)
        ax.text(mark_time, 1.01, mark_label, color="green", fontsize=12, fontweight="bold", ha="center", va="bottom", transform=ax.get_xaxis_transform(), clip_on=False)

sim_rate_reference_info = annotate_reference_profile_peak(
    ax1,
    SimRate_profile_matrix,
    sim_rate_profile_metadata,
    SimProfileDepth_rate,
    sim_rate_overlay_half_width_hours,
    sim_rate_scale_reference,
    reference_profile_time,
    "nanostrain/s",
    "Simulated strain-rate",
)
sim_strain_reference_info = annotate_reference_profile_peak(
    ax2,
    SimStrain_profile_matrix,
    sim_strain_profile_metadata,
    SimProfileDepth_strain,
    sim_strain_overlay_half_width_hours,
    sim_strain_scale_reference,
    reference_profile_time,
    "millistrain",
    "Simulated direct DDM strain",
)

ax1.set_xlim(sim_plot_start, sim_plot_end)
ax2.set_xlim(sim_plot_start, sim_plot_end)
ax2.set_xlabel("Time [UTC-7]")
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d\n%H:%M"))
plt.show()

print("Loaded sim_fault2 direct-DDM CSVs and plotted simulated waterfalls/profiles.")
print(f"Peak marker target profile time: {reference_profile_time}.")
if sim_rate_reference_info is not None:
    print(
        f"Simulated strain-rate peak near {sim_rate_reference_info['profile_time']}: "
        f"{sim_rate_reference_info['peak_value']:+.4g} {sim_rate_reference_info['unit']} "
        f"at MD {sim_rate_reference_info['peak_depth_ft']:.1f} ft"
    )
if sim_strain_reference_info is not None:
    print(
        f"Simulated strain peak near {sim_strain_reference_info['profile_time']}: "
        f"{sim_strain_reference_info['peak_value']:+.4g} {sim_strain_reference_info['unit']} "
        f"at MD {sim_strain_reference_info['peak_depth_ft']:.1f} ft"
    )
print(f"Strain-rate waterfall CSV: {sim_rate_waterfall_csv}")
print(f"Direct-DDM strain waterfall CSV: {sim_strain_waterfall_csv}")
print(f"Strain-rate profile CSV: {sim_rate_profile_csv}")
print(f"Strain-rate metadata CSV: {sim_rate_profile_meta_csv}")
print(f"Direct-DDM strain profile CSV: {sim_strain_profile_csv}")
print(f"Strain metadata CSV: {sim_strain_profile_meta_csv}")


# Observation versus Simulation

In [ ]:
# Observed/simulated profile comparison on observed and simulated waterfalls
# Run the Observation within V1 mds range cell first so LP_rate_plot, LP_strain_ref_mstrain,
# Depth_plot, and Time_plot are available. Run the modeling export cell before this if the
# simulated CSV files need to be refreshed.

from pathlib import Path
import matplotlib.dates as mdates

comparison_export_dir = Path(r"D:\OneDrive - Colorado School of Mines\Research\Fervo\Pengchao\code\profile_exports")
comparison_sim_export_dir = comparison_export_dir / "sim_fault2"
comparison_profile_interval = "4h"
comparison_profile_scale_multiplier = 10.0
comparison_rate_overlay_half_width_hours = 28.0
comparison_strain_overlay_half_width_hours = 28.0
comparison_rate_color_scale = (-0.3, 0.3)      # nanostrain/s
comparison_strain_color_scale = (-0.1, 0.1)    # millistrain
comparison_plot_start = pd.to_datetime("2025-02-24 11:00")
comparison_plot_end = pd.to_datetime("2025-03-03 22:00")
comparison_md_top = 10200
comparison_md_bottom = 10500
comparison_mark_times = pd.to_datetime([
    "2025-02-24 11:00",
    "2025-02-28 00:00",
    "2025-03-03 22:00",
])
comparison_mark_labels = ["T1", "T2", "T3"]
comparison_tick_times = pd.DatetimeIndex(pd.date_range(comparison_plot_start, comparison_plot_end, freq="24h"))
if comparison_plot_end not in comparison_tick_times:
    comparison_tick_times = comparison_tick_times.append(pd.DatetimeIndex([comparison_plot_end]))

required_observation_vars = ["LP_rate_plot", "LP_strain_ref_mstrain", "Depth_plot", "Time_plot"]
missing_observation_vars = [name for name in required_observation_vars if name not in globals()]
if missing_observation_vars:
    raise RuntimeError(
        "Run the 'Observation within V1 mds range' cell before this comparison cell. "
        f"Missing variables: {missing_observation_vars}"
    )


def latest_file(folder, pattern, exclude_metadata=False):
    files = list(folder.glob(pattern))
    if exclude_metadata:
        files = [path for path in files if not path.name.endswith("_metadata.csv")]
    if not files:
        raise FileNotFoundError(f"No file matching {pattern!r} found in {folder}")
    return max(files, key=lambda path: path.stat().st_mtime)


obs_rate_profile_csv = latest_file(
    comparison_export_dir,
    "strain_rate_4h_mean_profiles_*_T1_ref.csv",
    exclude_metadata=True,
)
obs_rate_profile_meta_csv = latest_file(
    comparison_export_dir,
    "strain_rate_4h_mean_profiles_*_T1_ref_metadata.csv",
)
obs_strain_profile_csv = latest_file(
    comparison_export_dir,
    "strain_4h_mean_profiles_*_T1_ref.csv",
    exclude_metadata=True,
)
obs_strain_profile_meta_csv = latest_file(
    comparison_export_dir,
    "strain_4h_mean_profiles_*_T1_ref_metadata.csv",
)
sim_rate_waterfall_csv = latest_file(comparison_sim_export_dir, "sim_rate_waterfall_*.csv")
sim_strain_waterfall_csv = latest_file(comparison_sim_export_dir, "sim_direct_strain_waterfall_*.csv")
sim_rate_profile_csv = latest_file(
    comparison_sim_export_dir,
    "sim_rate_4h_*.csv",
    exclude_metadata=True,
)
sim_rate_profile_meta_csv = latest_file(
    comparison_sim_export_dir,
    "sim_rate_4h_*_metadata.csv",
)
sim_strain_profile_csv = latest_file(
    comparison_sim_export_dir,
    "sim_direct_strain_4h_*.csv",
    exclude_metadata=True,
)
sim_strain_profile_meta_csv = latest_file(
    comparison_sim_export_dir,
    "sim_direct_strain_4h_*_metadata.csv",
)


def read_depth_time_csv(csv_path):
    df = pd.read_csv(csv_path)
    depth = df["measured_depth_ft"].to_numpy(dtype=float)
    time = pd.DatetimeIndex(pd.to_datetime(df.columns[1:]))
    data = df.iloc[:, 1:].to_numpy(dtype=float)
    return data, depth, time


def read_profile_csv(data_csv, metadata_csv):
    profile_df = pd.read_csv(data_csv)
    metadata = pd.read_csv(metadata_csv)
    depth = profile_df["measured_depth_ft"].to_numpy(dtype=float)
    profiles = profile_df.iloc[:, 1:].to_numpy(dtype=float)
    for col in ["window_start", "window_end", "profile_center_time"]:
        if col in metadata.columns:
            metadata[col] = pd.to_datetime(metadata[col])
    return profiles, metadata, depth


def combined_profile_abs_scale(*profile_matrices, multiplier=1.0):
    values = []
    for matrix in profile_matrices:
        if matrix.size:
            finite_values = np.abs(matrix[np.isfinite(matrix)])
            if finite_values.size:
                values.append(finite_values)
    if not values:
        return np.nan, np.nan
    combined = np.concatenate(values)
    p95 = np.nanpercentile(combined, 95)
    if not np.isfinite(p95) or p95 == 0:
        p95 = np.nanmax(combined)
    if not np.isfinite(p95) or p95 == 0:
        return np.nan, np.nan
    return p95, p95 * multiplier


def overlay_profiles_on_time_axis(
    ax,
    profile_matrix,
    profile_metadata,
    depth_values,
    half_width_hours,
    scale_reference_abs,
    color,
    linestyle,
    label,
    linewidth=0.9,
    alpha=0.95,
):
    if profile_matrix.size == 0 or not np.isfinite(scale_reference_abs) or scale_reference_abs == 0:
        return
    if "profile_center_time" not in profile_metadata.columns:
        raise KeyError("profile_metadata must contain 'profile_center_time'")
    scale_seconds_per_unit = half_width_hours * 3600.0 / scale_reference_abs
    for col_idx, row in profile_metadata.iterrows():
        profile = profile_matrix[:, col_idx]
        finite = np.isfinite(profile)
        if not finite.any():
            continue
        center_time = pd.Timestamp(row["profile_center_time"])
        x_times = center_time + pd.to_timedelta(profile[finite] * scale_seconds_per_unit, unit="s")
        ax.plot(
            x_times,
            depth_values[finite],
            color=color,
            linestyle=linestyle,
            linewidth=linewidth,
            alpha=alpha,
            zorder=5,
        )
    ax.plot([], [], color=color, linestyle=linestyle, linewidth=1.8, label=label)


def plot_waterfall(ax, data_matrix, time_values, depth_values, cmap_name, color_scale, cbar_label, title):
    x0 = mdates.date2num(pd.Timestamp(time_values[0]).to_pydatetime())
    x1 = mdates.date2num(pd.Timestamp(time_values[-1]).to_pydatetime())
    cmap = plt.get_cmap(cmap_name).copy()
    cmap.set_bad("0.82")
    im = ax.imshow(
        data_matrix,
        cmap=cmap,
        aspect="auto",
        vmin=color_scale[0],
        vmax=color_scale[1],
        extent=(x0, x1, depth_values[-1], depth_values[0]),
        interpolation="none",
    )
    ax.xaxis_date()
    ax.set_ylabel("Measured Depth (ft)")
    ax.set_ylim(comparison_md_bottom, comparison_md_top)
    ax.set_title(title)
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label(cbar_label)
    return im


def format_comparison_axis(ax):
    for label, mark_time in zip(comparison_mark_labels, comparison_mark_times):
        ax.axvline(mark_time, color="gold", linestyle="--", linewidth=1.4, alpha=0.95, zorder=4)
        ax.text(
            mark_time,
            1.01,
            label,
            color="gold",
            fontsize=11,
            fontweight="bold",
            ha="center",
            va="bottom",
            transform=ax.get_xaxis_transform(),
            clip_on=False,
        )
    ax.set_xlim(comparison_plot_start, comparison_plot_end)
    ax.set_xticks(comparison_tick_times)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d\n%H:%M"))
    ax.grid(color="white", alpha=0.18, linewidth=0.4)
    ax.legend(loc="lower right")


ObsRate_profile_matrix, obs_rate_profile_metadata, ObsProfileDepth_rate = read_profile_csv(
    obs_rate_profile_csv,
    obs_rate_profile_meta_csv,
)
ObsStrain_profile_matrix, obs_strain_profile_metadata, ObsProfileDepth_strain = read_profile_csv(
    obs_strain_profile_csv,
    obs_strain_profile_meta_csv,
)
SimRate_plot, SimDepth_rate, SimTime_plot = read_depth_time_csv(sim_rate_waterfall_csv)
SimStrain_plot, SimDepth_strain, SimStrainTime_plot = read_depth_time_csv(sim_strain_waterfall_csv)
SimRate_profile_matrix, sim_rate_profile_metadata, SimProfileDepth_rate = read_profile_csv(
    sim_rate_profile_csv,
    sim_rate_profile_meta_csv,
)
SimStrain_profile_matrix, sim_strain_profile_metadata, SimProfileDepth_strain = read_profile_csv(
    sim_strain_profile_csv,
    sim_strain_profile_meta_csv,
)

obs_rate_waterfall = np.asarray(LP_rate_plot, dtype=float)
obs_strain_waterfall = np.asarray(LP_strain_ref_mstrain, dtype=float)
obs_depth = np.asarray(Depth_plot, dtype=float)
obs_time = pd.DatetimeIndex(Time_plot)

rate_profile_p95, shared_rate_profile_scale = combined_profile_abs_scale(
    ObsRate_profile_matrix,
    SimRate_profile_matrix,
    multiplier=comparison_profile_scale_multiplier,
)
strain_profile_p95, shared_strain_profile_scale = combined_profile_abs_scale(
    ObsStrain_profile_matrix,
    SimStrain_profile_matrix,
    multiplier=comparison_profile_scale_multiplier,
)

# Figure 1: observed waterfalls as background, observed and simulated profiles overlaid.
fig_obs, (ax_obs_rate, ax_obs_strain) = plt.subplots(2, 1, figsize=(16, 9), sharex=True, constrained_layout=True)
plot_waterfall(
    ax_obs_rate,
    obs_rate_waterfall,
    obs_time,
    obs_depth,
    "bwr",
    comparison_rate_color_scale,
    "Strain rate (nanostrain/s)",
    "Observed strain-rate waterfall with observed and simulated 4-hour profiles",
)
overlay_profiles_on_time_axis(
    ax_obs_rate,
    ObsRate_profile_matrix,
    obs_rate_profile_metadata,
    ObsProfileDepth_rate,
    comparison_rate_overlay_half_width_hours,
    shared_rate_profile_scale,
    color="black",
    linestyle="-",
    label="Observed rate profiles",
)
overlay_profiles_on_time_axis(
    ax_obs_rate,
    SimRate_profile_matrix,
    sim_rate_profile_metadata,
    SimProfileDepth_rate,
    comparison_rate_overlay_half_width_hours,
    shared_rate_profile_scale,
    color="green",
    linestyle="--",
    label="Simulated rate profiles",
)
plot_waterfall(
    ax_obs_strain,
    obs_strain_waterfall,
    obs_time,
    obs_depth,
    "seismic",
    comparison_strain_color_scale,
    "Strain (millistrain)",
    "Observed T1-referenced strain waterfall with observed and simulated 4-hour profiles",
)
overlay_profiles_on_time_axis(
    ax_obs_strain,
    ObsStrain_profile_matrix,
    obs_strain_profile_metadata,
    ObsProfileDepth_strain,
    comparison_strain_overlay_half_width_hours,
    shared_strain_profile_scale,
    color="black",
    linestyle="-",
    label="Observed strain profiles",
)
overlay_profiles_on_time_axis(
    ax_obs_strain,
    SimStrain_profile_matrix,
    sim_strain_profile_metadata,
    SimProfileDepth_strain,
    comparison_strain_overlay_half_width_hours,
    shared_strain_profile_scale,
    color="green",
    linestyle="--",
    label="Simulated direct-DDM strain profiles",
)
for ax in (ax_obs_rate, ax_obs_strain):
    format_comparison_axis(ax)
ax_obs_strain.set_xlabel("Time [UTC-7]")
plt.show()

# Figure 2: simulated waterfalls as background, observed and simulated profiles overlaid.
fig_sim, (ax_sim_rate, ax_sim_strain) = plt.subplots(2, 1, figsize=(16, 9), sharex=True, constrained_layout=True)
plot_waterfall(
    ax_sim_rate,
    SimRate_plot,
    SimTime_plot,
    SimDepth_rate,
    "bwr",
    comparison_rate_color_scale,
    "Strain rate (nanostrain/s)",
    "Simulated strain-rate waterfall with observed and simulated 4-hour profiles",
)
overlay_profiles_on_time_axis(
    ax_sim_rate,
    ObsRate_profile_matrix,
    obs_rate_profile_metadata,
    ObsProfileDepth_rate,
    comparison_rate_overlay_half_width_hours,
    shared_rate_profile_scale,
    color="black",
    linestyle="-",
    label="Observed rate profiles",
)
overlay_profiles_on_time_axis(
    ax_sim_rate,
    SimRate_profile_matrix,
    sim_rate_profile_metadata,
    SimProfileDepth_rate,
    comparison_rate_overlay_half_width_hours,
    shared_rate_profile_scale,
    color="green",
    linestyle="--",
    label="Simulated rate profiles",
)
plot_waterfall(
    ax_sim_strain,
    SimStrain_plot,
    SimStrainTime_plot,
    SimDepth_strain,
    "seismic",
    comparison_strain_color_scale,
    "Strain (millistrain)",
    "Direct DDM T1-referenced strain waterfall with observed and simulated 4-hour profiles",
)
overlay_profiles_on_time_axis(
    ax_sim_strain,
    ObsStrain_profile_matrix,
    obs_strain_profile_metadata,
    ObsProfileDepth_strain,
    comparison_strain_overlay_half_width_hours,
    shared_strain_profile_scale,
    color="black",
    linestyle="-",
    label="Observed strain profiles",
)
overlay_profiles_on_time_axis(
    ax_sim_strain,
    SimStrain_profile_matrix,
    sim_strain_profile_metadata,
    SimProfileDepth_strain,
    comparison_strain_overlay_half_width_hours,
    shared_strain_profile_scale,
    color="green",
    linestyle="--",
    label="Simulated direct-DDM strain profiles",
)
for ax in (ax_sim_rate, ax_sim_strain):
    format_comparison_axis(ax)
ax_sim_strain.set_xlabel("Time [UTC-7]")
plt.show()

print("Observation/simulation versus comparison complete.")
print(f"Observed rate profile CSV: {obs_rate_profile_csv}")
print(f"Observed strain profile CSV: {obs_strain_profile_csv}")
print(f"Simulated rate profile CSV: {sim_rate_profile_csv}")
print(f"Simulated direct-DDM strain profile CSV: {sim_strain_profile_csv}")
print(f"Shared rate profile visual scale: +/-{shared_rate_profile_scale:.4g} nanostrain/s over +/-{comparison_rate_overlay_half_width_hours:g} hours")
print(f"Shared strain profile visual scale: +/-{shared_strain_profile_scale:.4g} millistrain over +/-{comparison_strain_overlay_half_width_hours:g} hours")


# 2nd Simulation within V1 mds range


In [ ]:
# Direct-DDM test-fault-length-2 waterfall/profile view from sim_fault2 CSV exports
# Run the modeling notebook's simulated export cell first. This cell reads those CSVs and
# plots simulated strain-rate and T1-referenced strain using the same 4-hour profile convention.

from pathlib import Path
import matplotlib.dates as mdates

sim_export_dir = Path(r"D:\OneDrive - Colorado School of Mines\Research\Fervo\Pengchao\code\profile_exports\sim_two_faults")
sim_profile_interval = "4h"
sim_rate_overlay_half_width_hours = 28.0
sim_strain_overlay_half_width_hours = 28.0
sim_profile_scale_multiplier = 10.0
sim_rate_color_scale = (-0.3, 0.3)      # nanostrain/s
sim_strain_color_scale = (-0.1, 0.1)    # millistrain
sim_plot_start = pd.to_datetime("2025-02-24 11:00")
sim_plot_end = pd.to_datetime("2025-03-03 22:00")
sim_md_top = 10200
sim_md_bottom = 10500
sim_time_marks = pd.to_datetime([
    "2025-02-24 11:00",
    "2025-02-28 00:00",
    "2025-03-03 22:00",
])
sim_mark_labels = ["T1", "T2", "T3"]
reference_profile_time = pd.to_datetime("2025-03-03 00:00")


def latest_file(pattern, exclude_metadata=False):
    files = list(sim_export_dir.glob(pattern))
    if exclude_metadata:
        files = [path for path in files if not path.name.endswith("_metadata.csv")]
    if not files:
        raise FileNotFoundError(
            f"No file matching {pattern!r} found in {sim_export_dir}. "
            "Run the simulated export cell in the modeling notebook first."
        )
    return max(files, key=lambda p: p.stat().st_mtime)


sim_rate_waterfall_csv = latest_file("two_fault_rate_waterfall_*.csv")
sim_strain_waterfall_csv = latest_file("two_fault_direct_strain_waterfall_*.csv")
sim_rate_profile_csv = latest_file("two_fault_rate_4h_*.csv", exclude_metadata=True)
sim_rate_profile_meta_csv = latest_file("two_fault_rate_4h_*_metadata.csv")
sim_strain_profile_csv = latest_file("two_fault_direct_strain_4h_*.csv", exclude_metadata=True)
sim_strain_profile_meta_csv = latest_file("two_fault_direct_strain_4h_*_metadata.csv")


def read_depth_time_csv(csv_path):
    df = pd.read_csv(csv_path)
    depth = df["measured_depth_ft"].to_numpy(dtype=float)
    time = pd.DatetimeIndex(pd.to_datetime(df.columns[1:]))
    data = df.iloc[:, 1:].to_numpy(dtype=float)
    return data, depth, time


def read_profile_csv(data_csv, metadata_csv):
    profile_df = pd.read_csv(data_csv)
    metadata = pd.read_csv(metadata_csv)
    depth = profile_df["measured_depth_ft"].to_numpy(dtype=float)
    profiles = profile_df.iloc[:, 1:].to_numpy(dtype=float)
    metadata["window_start"] = pd.to_datetime(metadata["window_start"])
    metadata["window_end"] = pd.to_datetime(metadata["window_end"])
    metadata["profile_center_time"] = pd.to_datetime(metadata["profile_center_time"])
    return profiles, metadata, depth


def get_profile_abs_scale(profile_matrix, multiplier=1.0):
    if profile_matrix.size == 0:
        return np.nan, np.nan
    profile_p95 = np.nanpercentile(np.abs(profile_matrix), 95)
    if not np.isfinite(profile_p95) or profile_p95 == 0:
        profile_p95 = np.nanmax(np.abs(profile_matrix))
    if not np.isfinite(profile_p95) or profile_p95 == 0:
        return np.nan, np.nan
    return profile_p95, profile_p95 * multiplier


def overlay_profiles_on_time_axis(ax, profile_matrix, profile_metadata, depth_values, half_width_hours, scale_reference_abs, color="black"):
    if profile_matrix.size == 0 or not np.isfinite(scale_reference_abs) or scale_reference_abs == 0:
        return
    scale_seconds_per_unit = half_width_hours * 3600.0 / scale_reference_abs
    for col_idx, row in profile_metadata.iterrows():
        profile = profile_matrix[:, col_idx]
        finite = np.isfinite(profile)
        if not finite.any():
            continue
        center_time = pd.Timestamp(row["profile_center_time"])
        x_times = center_time + pd.to_timedelta(profile[finite] * scale_seconds_per_unit, unit="s")
        ax.plot(x_times, depth_values[finite], color=color, linewidth=0.8, alpha=0.9, zorder=5)


def annotate_reference_profile_peak(
    ax,
    profile_matrix,
    profile_metadata,
    profile_depth,
    overlay_half_width_hours,
    scale_reference_abs,
    target_time,
    unit_label,
    profile_label,
):
    if profile_matrix.size == 0 or profile_metadata.empty:
        return None
    profile_times = pd.to_datetime(profile_metadata["profile_center_time"])
    profile_idx = int(np.argmin(np.abs(profile_times - target_time)))
    profile_values = profile_matrix[:, profile_idx]
    if profile_values.size == 0 or np.all(~np.isfinite(profile_values)):
        return None
    peak_idx = int(np.nanargmax(np.abs(profile_values)))
    peak_value = float(profile_values[peak_idx])
    peak_depth = float(profile_depth[peak_idx])
    peak_time = profile_times.iloc[profile_idx] + pd.to_timedelta(
        overlay_half_width_hours * 3600.0 * peak_value / scale_reference_abs,
        unit="s",
    )
    ax.scatter(
        peak_time,
        peak_depth,
        marker="*",
        color="yellow",
        edgecolor="black",
        linewidth=0.8,
        s=150,
        zorder=10,
    )
    ax.text(
        peak_time,
        peak_depth + 8,
        f"{peak_value:+.3g} {unit_label}",
        color="black",
        fontsize=9,
        fontweight="bold",
        ha="center",
        va="top",
        bbox=dict(facecolor="yellow", edgecolor="black", alpha=0.85, boxstyle="round,pad=0.2"),
        zorder=11,
    )
    return {
        "profile_label": profile_label,
        "profile_time": profile_times.iloc[profile_idx],
        "target_time": target_time,
        "peak_depth_ft": peak_depth,
        "peak_value": peak_value,
        "unit": unit_label,
    }


def plot_sim_waterfall(ax, data_matrix, time_values, depth_values, cmap_name, color_scale, cbar_label, title):
    x0 = mdates.date2num(pd.Timestamp(time_values[0]).to_pydatetime())
    x1 = mdates.date2num(pd.Timestamp(time_values[-1]).to_pydatetime())
    im = ax.imshow(
        data_matrix,
        cmap=cmap_name,
        aspect="auto",
        vmin=color_scale[0],
        vmax=color_scale[1],
        extent=(x0, x1, depth_values[-1], depth_values[0]),
        interpolation="none",
    )
    ax.xaxis_date()
    ax.set_ylabel("Measured Depth (ft)")
    ax.set_ylim(sim_md_bottom, sim_md_top)
    ax.set_title(title)
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label(cbar_label)
    return im


SimRate_plot, SimDepth_rate, SimTime_plot = read_depth_time_csv(sim_rate_waterfall_csv)
SimStrain_plot, SimDepth_strain, SimStrainTime_plot = read_depth_time_csv(sim_strain_waterfall_csv)
SimRate_profile_matrix, sim_rate_profile_metadata, SimProfileDepth_rate = read_profile_csv(sim_rate_profile_csv, sim_rate_profile_meta_csv)
SimStrain_profile_matrix, sim_strain_profile_metadata, SimProfileDepth_strain = read_profile_csv(sim_strain_profile_csv, sim_strain_profile_meta_csv)

sim_rate_p95, sim_rate_scale_reference = get_profile_abs_scale(SimRate_profile_matrix, sim_profile_scale_multiplier)
sim_strain_p95, sim_strain_scale_reference = get_profile_abs_scale(SimStrain_profile_matrix, sim_profile_scale_multiplier)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 9), sharex=True, constrained_layout=True)
plot_sim_waterfall(
    ax1,
    SimRate_plot,
    SimTime_plot,
    SimDepth_rate,
    "bwr",
    sim_rate_color_scale,
    "Strain rate (nanostrain/s)",
    "Simulated strain-rate waterfall with overlaid 4-hour mean profiles",
)
overlay_profiles_on_time_axis(
    ax1,
    SimRate_profile_matrix,
    sim_rate_profile_metadata,
    SimProfileDepth_rate,
    sim_rate_overlay_half_width_hours,
    sim_rate_scale_reference,
)
plot_sim_waterfall(
    ax2,
    SimStrain_plot,
    SimStrainTime_plot,
    SimDepth_strain,
    "seismic",
    sim_strain_color_scale,
    "Strain (millistrain)",
    "Direct DDM strain waterfall with overlaid 4-hour mean profiles (T1 referenced)",
)
overlay_profiles_on_time_axis(
    ax2,
    SimStrain_profile_matrix,
    sim_strain_profile_metadata,
    SimProfileDepth_strain,
    sim_strain_overlay_half_width_hours,
    sim_strain_scale_reference,
)

for mark_time, mark_label in zip(sim_time_marks, sim_mark_labels):
    for ax in (ax1, ax2):
        ax.axvline(mark_time, color="green", linestyle="--", linewidth=1.8, alpha=0.95, zorder=4)
        ax.text(mark_time, 1.01, mark_label, color="green", fontsize=12, fontweight="bold", ha="center", va="bottom", transform=ax.get_xaxis_transform(), clip_on=False)

sim_rate_reference_info = annotate_reference_profile_peak(
    ax1,
    SimRate_profile_matrix,
    sim_rate_profile_metadata,
    SimProfileDepth_rate,
    sim_rate_overlay_half_width_hours,
    sim_rate_scale_reference,
    reference_profile_time,
    "nanostrain/s",
    "Simulated strain-rate",
)
sim_strain_reference_info = annotate_reference_profile_peak(
    ax2,
    SimStrain_profile_matrix,
    sim_strain_profile_metadata,
    SimProfileDepth_strain,
    sim_strain_overlay_half_width_hours,
    sim_strain_scale_reference,
    reference_profile_time,
    "millistrain",
    "Two-fault direct DDM strain",
)

ax1.set_xlim(sim_plot_start, sim_plot_end)
ax2.set_xlim(sim_plot_start, sim_plot_end)
ax2.set_xlabel("Time [UTC-7]")
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d\n%H:%M"))
plt.show()

print("Loaded sim_two_faults direct-DDM CSVs and plotted simulated waterfalls/profiles.")
print(f"Peak marker target profile time: {reference_profile_time}.")
if sim_rate_reference_info is not None:
    print(
        f"Simulated strain-rate peak near {sim_rate_reference_info['profile_time']}: "
        f"{sim_rate_reference_info['peak_value']:+.4g} {sim_rate_reference_info['unit']} "
        f"at MD {sim_rate_reference_info['peak_depth_ft']:.1f} ft"
    )
if sim_strain_reference_info is not None:
    print(
        f"Simulated strain peak near {sim_strain_reference_info['profile_time']}: "
        f"{sim_strain_reference_info['peak_value']:+.4g} {sim_strain_reference_info['unit']} "
        f"at MD {sim_strain_reference_info['peak_depth_ft']:.1f} ft"
    )
print(f"Strain-rate waterfall CSV: {sim_rate_waterfall_csv}")
print(f"Direct-DDM strain waterfall CSV: {sim_strain_waterfall_csv}")
print(f"Strain-rate profile CSV: {sim_rate_profile_csv}")
print(f"Strain-rate metadata CSV: {sim_rate_profile_meta_csv}")
print(f"Direct-DDM strain profile CSV: {sim_strain_profile_csv}")
print(f"Strain metadata CSV: {sim_strain_profile_meta_csv}")


# 2nd Observation versus Simulation


In [ ]:
# Observed/simulated profile comparison on observed and simulated waterfalls
# Run the Observation within V1 mds range cell first so LP_rate_plot, LP_strain_ref_mstrain,
# Depth_plot, and Time_plot are available. Run the modeling export cell before this if the
# simulated CSV files need to be refreshed.

from pathlib import Path
import matplotlib.dates as mdates

comparison_export_dir = Path(r"D:\OneDrive - Colorado School of Mines\Research\Fervo\Pengchao\code\profile_exports")
comparison_sim_export_dir = comparison_export_dir / "sim_two_faults"
comparison_profile_interval = "4h"
comparison_profile_scale_multiplier = 10.0
comparison_rate_overlay_half_width_hours = 28.0
comparison_strain_overlay_half_width_hours = 28.0
comparison_rate_color_scale = (-0.3, 0.3)      # nanostrain/s
comparison_strain_color_scale = (-0.1, 0.1)    # millistrain
comparison_plot_start = pd.to_datetime("2025-02-24 11:00")
comparison_plot_end = pd.to_datetime("2025-03-03 22:00")
comparison_md_top = 10200
comparison_md_bottom = 10500
comparison_mark_times = pd.to_datetime([
    "2025-02-24 11:00",
    "2025-02-28 00:00",
    "2025-03-03 22:00",
])
comparison_mark_labels = ["T1", "T2", "T3"]
comparison_tick_times = pd.DatetimeIndex(pd.date_range(comparison_plot_start, comparison_plot_end, freq="24h"))
if comparison_plot_end not in comparison_tick_times:
    comparison_tick_times = comparison_tick_times.append(pd.DatetimeIndex([comparison_plot_end]))

required_observation_vars = ["LP_rate_plot", "LP_strain_ref_mstrain", "Depth_plot", "Time_plot"]
missing_observation_vars = [name for name in required_observation_vars if name not in globals()]
if missing_observation_vars:
    raise RuntimeError(
        "Run the 'Observation within V1 mds range' cell before this comparison cell. "
        f"Missing variables: {missing_observation_vars}"
    )


def latest_file(folder, pattern, exclude_metadata=False):
    files = list(folder.glob(pattern))
    if exclude_metadata:
        files = [path for path in files if not path.name.endswith("_metadata.csv")]
    if not files:
        raise FileNotFoundError(f"No file matching {pattern!r} found in {folder}")
    return max(files, key=lambda path: path.stat().st_mtime)


obs_rate_profile_csv = latest_file(
    comparison_export_dir,
    "strain_rate_4h_mean_profiles_*_T1_ref.csv",
    exclude_metadata=True,
)
obs_rate_profile_meta_csv = latest_file(
    comparison_export_dir,
    "strain_rate_4h_mean_profiles_*_T1_ref_metadata.csv",
)
obs_strain_profile_csv = latest_file(
    comparison_export_dir,
    "strain_4h_mean_profiles_*_T1_ref.csv",
    exclude_metadata=True,
)
obs_strain_profile_meta_csv = latest_file(
    comparison_export_dir,
    "strain_4h_mean_profiles_*_T1_ref_metadata.csv",
)
sim_rate_waterfall_csv = latest_file(comparison_sim_export_dir, "two_fault_rate_waterfall_*.csv")
sim_strain_waterfall_csv = latest_file(comparison_sim_export_dir, "two_fault_direct_strain_waterfall_*.csv")
sim_rate_profile_csv = latest_file(
    comparison_sim_export_dir,
    "two_fault_rate_4h_*.csv",
    exclude_metadata=True,
)
sim_rate_profile_meta_csv = latest_file(
    comparison_sim_export_dir,
    "two_fault_rate_4h_*_metadata.csv",
)
sim_strain_profile_csv = latest_file(
    comparison_sim_export_dir,
    "two_fault_direct_strain_4h_*.csv",
    exclude_metadata=True,
)
sim_strain_profile_meta_csv = latest_file(
    comparison_sim_export_dir,
    "two_fault_direct_strain_4h_*_metadata.csv",
)


def read_depth_time_csv(csv_path):
    df = pd.read_csv(csv_path)
    depth = df["measured_depth_ft"].to_numpy(dtype=float)
    time = pd.DatetimeIndex(pd.to_datetime(df.columns[1:]))
    data = df.iloc[:, 1:].to_numpy(dtype=float)
    return data, depth, time


def read_profile_csv(data_csv, metadata_csv):
    profile_df = pd.read_csv(data_csv)
    metadata = pd.read_csv(metadata_csv)
    depth = profile_df["measured_depth_ft"].to_numpy(dtype=float)
    profiles = profile_df.iloc[:, 1:].to_numpy(dtype=float)
    for col in ["window_start", "window_end", "profile_center_time"]:
        if col in metadata.columns:
            metadata[col] = pd.to_datetime(metadata[col])
    return profiles, metadata, depth


def combined_profile_abs_scale(*profile_matrices, multiplier=1.0):
    values = []
    for matrix in profile_matrices:
        if matrix.size:
            finite_values = np.abs(matrix[np.isfinite(matrix)])
            if finite_values.size:
                values.append(finite_values)
    if not values:
        return np.nan, np.nan
    combined = np.concatenate(values)
    p95 = np.nanpercentile(combined, 95)
    if not np.isfinite(p95) or p95 == 0:
        p95 = np.nanmax(combined)
    if not np.isfinite(p95) or p95 == 0:
        return np.nan, np.nan
    return p95, p95 * multiplier


def overlay_profiles_on_time_axis(
    ax,
    profile_matrix,
    profile_metadata,
    depth_values,
    half_width_hours,
    scale_reference_abs,
    color,
    linestyle,
    label,
    linewidth=0.9,
    alpha=0.95,
):
    if profile_matrix.size == 0 or not np.isfinite(scale_reference_abs) or scale_reference_abs == 0:
        return
    if "profile_center_time" not in profile_metadata.columns:
        raise KeyError("profile_metadata must contain 'profile_center_time'")
    scale_seconds_per_unit = half_width_hours * 3600.0 / scale_reference_abs
    for col_idx, row in profile_metadata.iterrows():
        profile = profile_matrix[:, col_idx]
        finite = np.isfinite(profile)
        if not finite.any():
            continue
        center_time = pd.Timestamp(row["profile_center_time"])
        x_times = center_time + pd.to_timedelta(profile[finite] * scale_seconds_per_unit, unit="s")
        ax.plot(
            x_times,
            depth_values[finite],
            color=color,
            linestyle=linestyle,
            linewidth=linewidth,
            alpha=alpha,
            zorder=5,
        )
    ax.plot([], [], color=color, linestyle=linestyle, linewidth=1.8, label=label)


def plot_waterfall(ax, data_matrix, time_values, depth_values, cmap_name, color_scale, cbar_label, title):
    x0 = mdates.date2num(pd.Timestamp(time_values[0]).to_pydatetime())
    x1 = mdates.date2num(pd.Timestamp(time_values[-1]).to_pydatetime())
    cmap = plt.get_cmap(cmap_name).copy()
    cmap.set_bad("0.82")
    im = ax.imshow(
        data_matrix,
        cmap=cmap,
        aspect="auto",
        vmin=color_scale[0],
        vmax=color_scale[1],
        extent=(x0, x1, depth_values[-1], depth_values[0]),
        interpolation="none",
    )
    ax.xaxis_date()
    ax.set_ylabel("Measured Depth (ft)")
    ax.set_ylim(comparison_md_bottom, comparison_md_top)
    ax.set_title(title)
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label(cbar_label)
    return im


def format_comparison_axis(ax):
    for label, mark_time in zip(comparison_mark_labels, comparison_mark_times):
        ax.axvline(mark_time, color="gold", linestyle="--", linewidth=1.4, alpha=0.95, zorder=4)
        ax.text(
            mark_time,
            1.01,
            label,
            color="gold",
            fontsize=11,
            fontweight="bold",
            ha="center",
            va="bottom",
            transform=ax.get_xaxis_transform(),
            clip_on=False,
        )
    ax.set_xlim(comparison_plot_start, comparison_plot_end)
    ax.set_xticks(comparison_tick_times)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d\n%H:%M"))
    ax.grid(color="white", alpha=0.18, linewidth=0.4)
    ax.legend(loc="lower right")


ObsRate_profile_matrix, obs_rate_profile_metadata, ObsProfileDepth_rate = read_profile_csv(
    obs_rate_profile_csv,
    obs_rate_profile_meta_csv,
)
ObsStrain_profile_matrix, obs_strain_profile_metadata, ObsProfileDepth_strain = read_profile_csv(
    obs_strain_profile_csv,
    obs_strain_profile_meta_csv,
)
SimRate_plot, SimDepth_rate, SimTime_plot = read_depth_time_csv(sim_rate_waterfall_csv)
SimStrain_plot, SimDepth_strain, SimStrainTime_plot = read_depth_time_csv(sim_strain_waterfall_csv)
SimRate_profile_matrix, sim_rate_profile_metadata, SimProfileDepth_rate = read_profile_csv(
    sim_rate_profile_csv,
    sim_rate_profile_meta_csv,
)
SimStrain_profile_matrix, sim_strain_profile_metadata, SimProfileDepth_strain = read_profile_csv(
    sim_strain_profile_csv,
    sim_strain_profile_meta_csv,
)

obs_rate_waterfall = np.asarray(LP_rate_plot, dtype=float)
obs_strain_waterfall = np.asarray(LP_strain_ref_mstrain, dtype=float)
obs_depth = np.asarray(Depth_plot, dtype=float)
obs_time = pd.DatetimeIndex(Time_plot)

rate_profile_p95, shared_rate_profile_scale = combined_profile_abs_scale(
    ObsRate_profile_matrix,
    SimRate_profile_matrix,
    multiplier=comparison_profile_scale_multiplier,
)
strain_profile_p95, shared_strain_profile_scale = combined_profile_abs_scale(
    ObsStrain_profile_matrix,
    SimStrain_profile_matrix,
    multiplier=comparison_profile_scale_multiplier,
)

# Figure 1: observed waterfalls as background, observed and simulated profiles overlaid.
fig_obs, (ax_obs_rate, ax_obs_strain) = plt.subplots(2, 1, figsize=(16, 9), sharex=True, constrained_layout=True)
plot_waterfall(
    ax_obs_rate,
    obs_rate_waterfall,
    obs_time,
    obs_depth,
    "bwr",
    comparison_rate_color_scale,
    "Strain rate (nanostrain/s)",
    "Observed strain-rate waterfall with observed and simulated 4-hour profiles",
)
overlay_profiles_on_time_axis(
    ax_obs_rate,
    ObsRate_profile_matrix,
    obs_rate_profile_metadata,
    ObsProfileDepth_rate,
    comparison_rate_overlay_half_width_hours,
    shared_rate_profile_scale,
    color="black",
    linestyle="-",
    label="Observed rate profiles",
)
overlay_profiles_on_time_axis(
    ax_obs_rate,
    SimRate_profile_matrix,
    sim_rate_profile_metadata,
    SimProfileDepth_rate,
    comparison_rate_overlay_half_width_hours,
    shared_rate_profile_scale,
    color="green",
    linestyle="--",
    label="Two-fault simulated rate profiles",
)
plot_waterfall(
    ax_obs_strain,
    obs_strain_waterfall,
    obs_time,
    obs_depth,
    "seismic",
    comparison_strain_color_scale,
    "Strain (millistrain)",
    "Observed T1-referenced strain waterfall with observed and simulated 4-hour profiles",
)
overlay_profiles_on_time_axis(
    ax_obs_strain,
    ObsStrain_profile_matrix,
    obs_strain_profile_metadata,
    ObsProfileDepth_strain,
    comparison_strain_overlay_half_width_hours,
    shared_strain_profile_scale,
    color="black",
    linestyle="-",
    label="Observed strain profiles",
)
overlay_profiles_on_time_axis(
    ax_obs_strain,
    SimStrain_profile_matrix,
    sim_strain_profile_metadata,
    SimProfileDepth_strain,
    comparison_strain_overlay_half_width_hours,
    shared_strain_profile_scale,
    color="green",
    linestyle="--",
    label="Two-fault two-fault simulated direct-DDM strain profiles",
)
for ax in (ax_obs_rate, ax_obs_strain):
    format_comparison_axis(ax)
ax_obs_strain.set_xlabel("Time [UTC-7]")
plt.show()

# Figure 2: simulated waterfalls as background, observed and simulated profiles overlaid.
fig_sim, (ax_sim_rate, ax_sim_strain) = plt.subplots(2, 1, figsize=(16, 9), sharex=True, constrained_layout=True)
plot_waterfall(
    ax_sim_rate,
    SimRate_plot,
    SimTime_plot,
    SimDepth_rate,
    "bwr",
    comparison_rate_color_scale,
    "Strain rate (nanostrain/s)",
    "Simulated strain-rate waterfall with observed and simulated 4-hour profiles",
)
overlay_profiles_on_time_axis(
    ax_sim_rate,
    ObsRate_profile_matrix,
    obs_rate_profile_metadata,
    ObsProfileDepth_rate,
    comparison_rate_overlay_half_width_hours,
    shared_rate_profile_scale,
    color="black",
    linestyle="-",
    label="Observed rate profiles",
)
overlay_profiles_on_time_axis(
    ax_sim_rate,
    SimRate_profile_matrix,
    sim_rate_profile_metadata,
    SimProfileDepth_rate,
    comparison_rate_overlay_half_width_hours,
    shared_rate_profile_scale,
    color="green",
    linestyle="--",
    label="Two-fault simulated rate profiles",
)
plot_waterfall(
    ax_sim_strain,
    SimStrain_plot,
    SimStrainTime_plot,
    SimDepth_strain,
    "seismic",
    comparison_strain_color_scale,
    "Strain (millistrain)",
    "Direct DDM T1-referenced strain waterfall with observed and simulated 4-hour profiles",
)
overlay_profiles_on_time_axis(
    ax_sim_strain,
    ObsStrain_profile_matrix,
    obs_strain_profile_metadata,
    ObsProfileDepth_strain,
    comparison_strain_overlay_half_width_hours,
    shared_strain_profile_scale,
    color="black",
    linestyle="-",
    label="Observed strain profiles",
)
overlay_profiles_on_time_axis(
    ax_sim_strain,
    SimStrain_profile_matrix,
    sim_strain_profile_metadata,
    SimProfileDepth_strain,
    comparison_strain_overlay_half_width_hours,
    shared_strain_profile_scale,
    color="green",
    linestyle="--",
    label="Two-fault two-fault simulated direct-DDM strain profiles",
)
for ax in (ax_sim_rate, ax_sim_strain):
    format_comparison_axis(ax)
ax_sim_strain.set_xlabel("Time [UTC-7]")
plt.show()

print("Observation/simulation versus comparison complete.")
print(f"Observed rate profile CSV: {obs_rate_profile_csv}")
print(f"Observed strain profile CSV: {obs_strain_profile_csv}")
print(f"Simulated rate profile CSV: {sim_rate_profile_csv}")
print(f"Simulated direct-DDM strain profile CSV: {sim_strain_profile_csv}")
print(f"Shared rate profile visual scale: +/-{shared_rate_profile_scale:.4g} nanostrain/s over +/-{comparison_rate_overlay_half_width_hours:g} hours")
print(f"Shared strain profile visual scale: +/-{shared_strain_profile_scale:.4g} millistrain over +/-{comparison_strain_overlay_half_width_hours:g} hours")
